# Mini-projet PySpark


## Partie 1 — Initialisation de Spark

In [129]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MultiSourceDataCleaning")
    .master("local[*]")
    .getOrCreate())

sc = spark.sparkContext


In [130]:
print("Nom de l'application :", sc.appName)
print("Mode d'exécution Spark :", sc.master)
print("Version de Spark :", spark.version)
print("Niveau de parallélisme par défaut :", sc.defaultParallelism)

Nom de l'application : MultiSourceDataCleaning
Mode d'exécution Spark : local[*]
Version de Spark : 3.5.1
Niveau de parallélisme par défaut : 10


In [131]:
#.master("local[*]") : Spark tourne localement dans le conteneur, en utilisant tous les cœurs disponibles. 


## Partie 2 — Chargement des données

### Étape 2 : chargement de la source relationnelle : CSV → PostgreSQL → Spark avec JDBC

#### Crée la configuration JDBC  ;  Jupyter et PostgreSQL communiquent à l’intérieur du réseau Docker.

In [132]:
jdbc_url = "jdbc:postgresql://postgres_tp:5432/ecommerce"

jdbc_properties = {
    "user": "admin",
    "password": "admin",
    "driver": "org.postgresql.Driver"}

In [133]:
customers_df = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=jdbc_properties)

In [134]:
print("TABLE : customers")

customers_df.printSchema()
customers_df.show(5, truncate=False)

print("Nombre de lignes :", customers_df.count())
print("Nombre de partitions :", customers_df.rdd.getNumPartitions())
print("Types détectés :", customers_df.dtypes)

TABLE : customers
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- created_at: timestamp (nullable = true)

+-----------+----------+---------+-----------------+-------------+--------+---------------+----------+-------------------+
|customer_id|first_name|last_name|email            |phone        |city    |country        |birth_date|created_at         |
+-----------+----------+---------+-----------------+-------------+--------+---------------+----------+-------------------+
|c001       |Prenom1   |Nom1     |user1@example.com|NULL         |Paris   |France         |1990/12/01|2025-01-02 00:00:00|
| C002      |Prenom2   |Nom2     |user2@example.com|0033612345678|Paris   |France         |1990-12-01|2025-01-03 00:00:00|

In [135]:
orders_df = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    properties=jdbc_properties)

In [136]:
print("TABLE : orders")

orders_df.printSchema()
orders_df.show(5, truncate=False)

print("Nombre de lignes :", orders_df.count())
print("Nombre de partitions :", orders_df.rdd.getNumPartitions())
print("Types détectés :", orders_df.dtypes)

TABLE : orders
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- total_amount: decimal(38,18) (nullable = true)
 |-- currency: string (nullable = true)

+--------+-----------+----------+---------+--------------+----------------------+--------+
|order_id|customer_id|order_date|status   |payment_method|total_amount          |currency|
+--------+-----------+----------+---------+--------------+----------------------+--------+
|ord-0001|C999       |2026-03-01|completed|CARD          |100.500000000000000000|MAD     |
| ORD2   |c004       |2026-03-01|paid     |WIRE          |-20.000000000000000000|NULL    |
| ORD3   |cust-26    |01/03/2026|PAID     |PAYPAL        |100.500000000000000000|NULL    |
|ord-0004|c004       |2026/03/01|annulée  |WIRE          |100.500000000000000000|EUR     |
|ORD-0005| C002      |2026-03-01|PAID    

In [137]:
order_items_df = spark.read.jdbc(
    url=jdbc_url,
    table="order_items",
    properties=jdbc_properties)

In [138]:
print("TABLE : order_items")

order_items_df.printSchema()
order_items_df.show(5, truncate=False)

print("Nombre de lignes :", order_items_df.count())
print("Nombre de partitions :", order_items_df.rdd.getNumPartitions())
print("Types détectés :", order_items_df.dtypes)

TABLE : order_items
root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(38,18) (nullable = true)
 |-- discount: decimal(38,18) (nullable = true)

+--------+----------+--------+----------------------+--------------------+
|order_id|product_id|quantity|unit_price            |discount            |
+--------+----------+--------+----------------------+--------------------+
|ord-0036|P012      |-1      |100.000000000000000000|0.100000000000000000|
|ord-0028|P001      |-1      |10.000000000000000000 |0.100000000000000000|
|ORD-0031|P002      |1       |25.000000000000000000 |0.200000000000000000|
| ORD21  |P010      |2       |10.000000000000000000 |0.000000000000000000|
|ord-0040|P012      |-1      |100.000000000000000000|1.200000000000000000|
+--------+----------+--------+----------------------+--------------------+
only showing top 5 rows

Nombre de lignes : 80
Nombre de partitions : 1
Types 

In [139]:
products_df = spark.read.jdbc(
    url=jdbc_url,
    table="products",
    properties=jdbc_properties)

In [140]:
print("TABLE : products")

order_items_df.printSchema()
order_items_df.show(5, truncate=False)

print("Nombre de lignes :", products_df.count())
print("Nombre de partitions :", products_df.rdd.getNumPartitions())
print("Types détectés :", products_df.dtypes)

TABLE : products
root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(38,18) (nullable = true)
 |-- discount: decimal(38,18) (nullable = true)

+--------+----------+--------+----------------------+--------------------+
|order_id|product_id|quantity|unit_price            |discount            |
+--------+----------+--------+----------------------+--------------------+
|ord-0036|P012      |-1      |100.000000000000000000|0.100000000000000000|
|ord-0028|P001      |-1      |10.000000000000000000 |0.100000000000000000|
|ORD-0031|P002      |1       |25.000000000000000000 |0.200000000000000000|
| ORD21  |P010      |2       |10.000000000000000000 |0.000000000000000000|
|ord-0040|P012      |-1      |100.000000000000000000|1.200000000000000000|
+--------+----------+--------+----------------------+--------------------+
only showing top 5 rows

Nombre de lignes : 15
Nombre de partitions : 1
Types dét

### Étape 3 : chargement de la collection MongoDB reviews 

In [141]:
reviews_df = (
    spark.read
    .format("mongodb")
    .option("spark.mongodb.read.connection.uri", "mongodb://mongo_tp:27017/ecommerce.reviews")
    .load())

In [142]:
print("Schéma de la collection reviews :")
reviews_df.printSchema()

print("Nombre de documents :", reviews_df.count())

print("Cinq premiers documents :")
reviews_df.show(5, truncate=False)

Schéma de la collection reviews :
root
 |-- _id: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- customerId: string (nullable = true)
 |-- orderId: string (nullable = true)
 |-- productId: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- reviewDate: string (nullable = true)
 |-- verifiedPurchase: string (nullable = true)

Nombre de documents : 30
Cinq premiers documents :
+------+----------------+----------+--------+---------+------+-------------------+----------------+
|_id   |comment         |customerId|orderId |productId|rating|reviewDate         |verifiedPurchase|
+------+----------------+----------+--------+---------+------+-------------------+----------------+
|REV001|Très bon produit|cust-26   | ORD33  |P009     |7     |2026-03-12T10:25:00|yes             |
|REV002|Très bon produit| C019     | ORD33  |P013     |3     |2026-03-12T10:25:00|yes             |
|REV003|Très bon produit| C018     | ORD27  |P004     |5     |2026-03-12T10:25:00|0 

In [143]:
from pyspark.sql.functions import col, sum as spark_sum, when

null_counts_reviews = reviews_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in reviews_df.columns
])

null_counts_reviews.show(truncate=False)

+---+-------+----------+-------+---------+------+----------+----------------+
|_id|comment|customerId|orderId|productId|rating|reviewDate|verifiedPurchase|
+---+-------+----------+-------+---------+------+----------+----------------+
|0  |0      |0         |0      |0        |0     |0         |0               |
+---+-------+----------+-------+---------+------+----------+----------------+



### Étape 4 : chargement des fichiers JSON

In [144]:
# Le schéma explicite

from pyspark.sql.types import StructType, StructField, StringType

delivery_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("location", StructType([
        StructField("city", StringType(), True),
        StructField("country", StringType(), True)
    ]), True),
    StructField("carrier", StructType([
        StructField("id", StringType(), True),
        StructField("name", StringType(), True)
    ]), True)])

In [145]:
# lecture avec schéma inféré

delivery_events_infer_df = (
    spark.read
    .option("multiLine", True)
    .json("/data/delivery_events.json"))

print("Schéma inféré automatiquement :")
delivery_events_infer_df.printSchema()

print("Cinq premières lignes :")
delivery_events_infer_df.show(5, truncate=False)

print("Nombre de lignes :", delivery_events_infer_df.count())

Schéma inféré automatiquement :
root
 |-- carrier: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |-- order_id: string (nullable = true)

Cinq premières lignes :
+------------+--------+-------------------+-------------+------------------+--------+
|carrier     |event_id|event_timestamp    |event_type   |location          |order_id|
+------------+--------+-------------------+-------------+------------------+--------+
|{DHL01, UPS}|EVT001  |2026-03-17T08:30:00|X            |{PARIS, France}   |ord-0036|
|{UPS01, DHL}|EVT002  |2026-03-04T08:30:00|X            |{Paris, France}   |ord-0029|
|{DHL01, UPS}|EVT003  |2026-03-12T08:30:00|X            |{Paris, France}   | ORD25  |
|{DHL01, 

In [146]:
# lecture avec schéma explicite

delivery_events_df = (
    spark.read
    .schema(delivery_schema)
    .option("multiLine", True)
    .json("/data/delivery_events.json"))

print("Schéma défini manuellement :")
delivery_events_df.printSchema()

print("Cinq premières lignes :")
delivery_events_df.show(5, truncate=False)

print("Nombre de lignes :", delivery_events_df.count())
print("Nombre de partitions :", delivery_events_df.rdd.getNumPartitions())
print("Types détectés :", delivery_events_df.dtypes)

Schéma défini manuellement :
root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |-- carrier: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)

Cinq premières lignes :
+--------+--------+-------------+-------------------+------------------+------------+
|event_id|order_id|event_type   |event_timestamp    |location          |carrier     |
+--------+--------+-------------+-------------------+------------------+------------+
|EVT001  |ord-0036|X            |2026-03-17T08:30:00|{PARIS, France}   |{DHL01, UPS}|
|EVT002  |ord-0029|X            |2026-03-04T08:30:00|{Paris, France}   |{UPS01, DHL}|
|EVT003  | ORD25  |X            |2026-03-12T08:30:00|{Paris, France}   |{DHL01, UPS}|
|EVT004  |OR

##### Comparaison entre schéma inféré et schéma explicite

L'inférence automatique permet à Spark de deviner les types des colonnes à partir du contenu du fichier JSON.

Cependant, dans un projet de nettoyage de données, un schéma explicite est préférable car :
- il évite les erreurs d'interprétation des types ;
- il garantit une structure stable même si certaines lignes contiennent des valeurs nulles ;
- il améliore la lisibilité du programme ;
- il permet de mieux contrôler les champs imbriqués ;
- il rend le traitement plus fiable en production.

## Partie 3 — Audit initial de la qualité

### Étape 5 : création d’un rapport de qualité

In [147]:
from pyspark.sql.functions import col, trim, length, when, lit
from pyspark.sql.types import StringType

In [148]:
# pour compter les doublons complets

def afficher_doublons_complets(nom_source, df):
    total = df.count()
    distinct_total = df.dropDuplicates().count()
    doublons = total - distinct_total
    
    print("Source :", nom_source)
    print("Nombre total de lignes :", total)
    print("Nombre de lignes distinctes :", distinct_total)
    print("Nombre de doublons complets :", doublons)
    print("-" * 60)

In [149]:
afficher_doublons_complets("customers", customers_df)
afficher_doublons_complets("orders", orders_df)
afficher_doublons_complets("order_items", order_items_df)
afficher_doublons_complets("products", products_df)
afficher_doublons_complets("reviews", reviews_df)
afficher_doublons_complets("delivery_events", delivery_events_df)

Source : customers
Nombre total de lignes : 30
Nombre de lignes distinctes : 30
Nombre de doublons complets : 0
------------------------------------------------------------
Source : orders
Nombre total de lignes : 40
Nombre de lignes distinctes : 40
Nombre de doublons complets : 0
------------------------------------------------------------
Source : order_items
Nombre total de lignes : 80
Nombre de lignes distinctes : 80
Nombre de doublons complets : 0
------------------------------------------------------------
Source : products
Nombre total de lignes : 15
Nombre de lignes distinctes : 15
Nombre de doublons complets : 0
------------------------------------------------------------
Source : reviews
Nombre total de lignes : 30
Nombre de lignes distinctes : 30
Nombre de doublons complets : 0
------------------------------------------------------------
Source : delivery_events
Nombre total de lignes : 60
Nombre de lignes distinctes : 60
Nombre de doublons complets : 0
---------------------

In [150]:
#règles simples de valeurs invalides, juste pour repérer les problèmes.

invalid_rules = {
    "customers": {
        "customer_id": col("customer_id").isNull() | (trim(col("customer_id")) == ""),
        "email": ~col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"),
        "phone": col("phone").isNull() | (trim(col("phone")) == ""),
        "birth_date": col("birth_date").isNull() | (trim(col("birth_date")) == "")
    },
    
    "orders": {
        "order_id": col("order_id").isNull() | (trim(col("order_id")) == ""),
        "customer_id": col("customer_id").isNull() | (trim(col("customer_id")) == ""),
        "total_amount": col("total_amount").isNull() | (col("total_amount") <= 0),
        "currency": col("currency").isNull() | (~col("currency").isin("EUR", "USD", "GBP", "MAD")),
        "status": col("status").isNull() | (trim(col("status")) == "")
    },
    
    "order_items": {
        "order_id": col("order_id").isNull() | (trim(col("order_id")) == ""),
        "product_id": col("product_id").isNull() | (trim(col("product_id")) == ""),
        "quantity": col("quantity").isNull() | (col("quantity") <= 0),
        "unit_price": col("unit_price").isNull() | (col("unit_price") < 0),
        "discount": col("discount").isNotNull() & ((col("discount") < 0) | (col("discount") > 1))
    },
    
    "products": {
        "product_id": col("product_id").isNull() | (trim(col("product_id")) == ""),
        "product_name": col("product_name").isNull() | (trim(col("product_name")) == ""),
        "current_price": col("current_price").isNull() | (col("current_price") <= 0)
    },
    
    "reviews": {
        "customerId": col("customerId").isNull() | (trim(col("customerId")) == ""),
        "productId": col("productId").isNull() | (trim(col("productId")) == ""),
        "orderId": col("orderId").isNull() | (trim(col("orderId")) == ""),
        "rating": col("rating").isNull() | (col("rating") < 1) | (col("rating") > 5),
        "reviewDate": col("reviewDate").isNull() | (trim(col("reviewDate")) == ""),
        "verifiedPurchase": col("verifiedPurchase").isNull() | (~col("verifiedPurchase").isin("true", "false", "1", "0", "yes", "no"))
    },
    
    "delivery_events": {
        "event_id": col("event_id").isNull() | (trim(col("event_id")) == ""),
        "order_id": col("order_id").isNull() | (trim(col("order_id")) == ""),
        "event_type": col("event_type").isNull() | (~col("event_type").isin(
            "ORDER_CREATED", "PREPARING", "SHIPPED", "IN_TRANSIT", "DELIVERED", "RETURNED"
        )),
        "event_timestamp": col("event_timestamp").isNull() | (trim(col("event_timestamp")) == "")
    }
}

In [151]:
# le DataFrame  

def construire_rapport_qualite(nom_source, df, regles_invalides):
    total_lignes = df.count()
    lignes_rapport = []
    
    for colonne, type_colonne in df.dtypes:
        nombre_nulles = df.filter(col(colonne).isNull()).count()
        
        if type_colonne == "string":
            nombre_chaines_vides = df.filter(
                col(colonne).isNotNull() & (trim(col(colonne)) == "")
            ).count()
        else:
            nombre_chaines_vides = 0
        
        if colonne in regles_invalides:
            nombre_invalides = df.filter(regles_invalides[colonne]).count()
        else:
            nombre_invalides = 0
        
        pourcentage_nulles = round((nombre_nulles / total_lignes) * 100, 2) if total_lignes > 0 else 0
        
        lignes_rapport.append((
            nom_source,
            colonne,
            total_lignes,
            nombre_nulles,
            pourcentage_nulles,
            nombre_chaines_vides,
            nombre_invalides
        ))
    
    colonnes = [
        "source",
        "colonne",
        "nombre_lignes",
        "nombre_nulles",
        "pourcentage_nulles",
        "nombre_chaines_vides",
        "nombre_invalides"
    ]
    
    return spark.createDataFrame(lignes_rapport, colonnes)

In [152]:
#pour créer les rapports pour chaque source

rapport_customers = construire_rapport_qualite(
    "customers",
    customers_df,
    invalid_rules["customers"]
)

rapport_orders = construire_rapport_qualite(
    "orders",
    orders_df,
    invalid_rules["orders"]
)

rapport_order_items = construire_rapport_qualite(
    "order_items",
    order_items_df,
    invalid_rules["order_items"]
)

rapport_products = construire_rapport_qualite(
    "products",
    products_df,
    invalid_rules["products"]
)

rapport_reviews = construire_rapport_qualite(
    "reviews",
    reviews_df,
    invalid_rules["reviews"]
)

rapport_delivery_events = construire_rapport_qualite(
    "delivery_events",
    delivery_events_df,
    invalid_rules["delivery_events"]
)

In [153]:
#pour fusionner le rapport qualité global

data_quality_report = (
    rapport_customers
    .unionByName(rapport_orders)
    .unionByName(rapport_order_items)
    .unionByName(rapport_products)
    .unionByName(rapport_reviews)
    .unionByName(rapport_delivery_events)
)

data_quality_report.show(100, truncate=False)

+---------------+----------------+-------------+-------------+------------------+--------------------+----------------+
|source         |colonne         |nombre_lignes|nombre_nulles|pourcentage_nulles|nombre_chaines_vides|nombre_invalides|
+---------------+----------------+-------------+-------------+------------------+--------------------+----------------+
|customers      |customer_id     |30           |0            |0.0               |0                   |0               |
|customers      |first_name      |30           |0            |0.0               |0                   |0               |
|customers      |last_name       |30           |0            |0.0               |0                   |0               |
|customers      |email           |30           |0            |0.0               |0                   |4               |
|customers      |phone           |30           |5            |16.67             |0                   |5               |
|customers      |city            |30    

In [154]:
#enregistrer le rapport qualité

data_quality_report.write.mode("overwrite").parquet("/output/data_quality_report")

In [155]:
rapport_recharge = spark.read.parquet("/output/data_quality_report")
rapport_recharge.show(100, truncate=False)

+---------------+----------------+-------------+-------------+------------------+--------------------+----------------+
|source         |colonne         |nombre_lignes|nombre_nulles|pourcentage_nulles|nombre_chaines_vides|nombre_invalides|
+---------------+----------------+-------------+-------------+------------------+--------------------+----------------+
|delivery_events|event_timestamp |60           |0            |0.0               |0                   |0               |
|delivery_events|event_type      |60           |0            |0.0               |0                   |11              |
|delivery_events|location        |60           |0            |0.0               |0                   |0               |
|delivery_events|order_id        |60           |0            |0.0               |0                   |0               |
|delivery_events|event_id        |60           |0            |0.0               |0                   |0               |
|reviews        |verifiedPurchase|30    

##### Le rapport de qualité a été enregistré au format Parquet dans `/output/data_quality_report`, qui correspond au dossier `output/data_quality_report` du projet local. Il a ensuite été rechargé dans Spark afin de vérifier que l'écriture s'est bien déroulée.

## Partie 4 — Nettoyage de la table des clients

#### Étape 6 : normalisation des identifiants clients

In [156]:
from pyspark.sql.functions import (
    col, trim, upper, regexp_replace, regexp_extract,
    lpad, concat, lit, when, current_timestamp,
    struct, to_json)

In [157]:
customers_id_cleaning = (
    customers_df
    .withColumn("customer_id_raw", col("customer_id"))
    
    # 1. Suppression des espaces début/fin + majuscules
    .withColumn("customer_id_step1", upper(trim(col("customer_id"))))
    
    # 2. Suppression des tirets, underscores et espaces internes
    .withColumn("customer_id_step2", regexp_replace(col("customer_id_step1"), "[-_\\s]", ""))
    
    # 3. Remplacement du préfixe CUST par C
    .withColumn("customer_id_step3", regexp_replace(col("customer_id_step2"), "^CUST", "C"))
    
    # 4. Extraction de la partie numérique
    .withColumn("customer_id_number", regexp_extract(col("customer_id_step3"), "^C0*([0-9]+)$", 1))
    
    # 5. Création du format final C000001
    .withColumn(
        "customer_id_clean",
        when(
            col("customer_id_number") != "",
            concat(lit("C"), lpad(col("customer_id_number"), 6, "0"))
        ).otherwise(None)
    )
)

In [158]:
customers_id_cleaning.select(
    "customer_id",
    "customer_id_step1",
    "customer_id_step2",
    "customer_id_step3",
    "customer_id_number",
    "customer_id_clean"
).show(30, truncate=False)

+-----------+-----------------+-----------------+-----------------+------------------+-----------------+
|customer_id|customer_id_step1|customer_id_step2|customer_id_step3|customer_id_number|customer_id_clean|
+-----------+-----------------+-----------------+-----------------+------------------+-----------------+
|c001       |C001             |C001             |C001             |1                 |C000001          |
| C002      |C002             |C002             |C002             |2                 |C000002          |
|C003       |C003             |C003             |C003             |3                 |C000003          |
|c004       |C004             |C004             |C004             |4                 |C000004          |
|C005       |C005             |C005             |C005             |5                 |C000005          |
| C006      |C006             |C006             |C006             |6                 |C000006          |
| C007      |C007             |C007             |C007  

In [159]:
#la table de rejet des identifiants impossibles

customers_rejects_step6 = (
    customers_id_cleaning
    .filter(col("customer_id_clean").isNull())
    .select(
        lit("customers").alias("source"),
        lit("Identifiant client impossible à normaliser").alias("rejection_reason"),
        current_timestamp().alias("rejection_timestamp"),
        to_json(struct([col(c) for c in customers_df.columns])).alias("original_data")))

customers_rejects_step6.show(truncate=False)

+------+----------------+-------------------+-------------+
|source|rejection_reason|rejection_timestamp|original_data|
+------+----------------+-------------------+-------------+
+------+----------------+-------------------+-------------+



In [160]:
customers_step6 = (
    customers_id_cleaning
    .filter(col("customer_id_clean").isNotNull())
    .drop(
        "customer_id_raw",
        "customer_id_step1",
        "customer_id_step2",
        "customer_id_step3",
        "customer_id_number"))

customers_step6.select(
    "customer_id",
    "customer_id_clean",
    "first_name",
    "last_name",
    "email").show(20, truncate=False)

+-----------+-----------------+----------+---------+------------------+
|customer_id|customer_id_clean|first_name|last_name|email             |
+-----------+-----------------+----------+---------+------------------+
|c001       |C000001          |Prenom1   |Nom1     |user1@example.com |
| C002      |C000002          |Prenom2   |Nom2     |user2@example.com |
|C003       |C000003          |Prenom3   |Nom3     |user3@example.com |
|c004       |C000004          |Prenom4   |Nom4     |user4@example.com |
|C005       |C000005          |Prenom5   |Nom5     |user5@example.com |
| C006      |C000006          |Prenom6   |Nom6     |user6@example.com |
| C007      |C000007          |Prenom7   |Nom7     |user7example.com  |
|C008       |C000008          |Prenom8   |Nom8     |user8@example.com |
|C009       |C000009          |Prenom9   |Nom9     |user9@example.com |
|C010       |C000010          |Prenom10  |Nom10    |user10@example.com|
|C011       |C000011          |Prenom11  |Nom11    |user11@examp

In [161]:
print("Nombre de lignes initiales customers :", customers_df.count())
print("Nombre de lignes avec identifiant normalisé :", customers_step6.count())
print("Nombre de rejets étape 6 :", customers_rejects_step6.count())


Nombre de lignes initiales customers : 30
Nombre de lignes avec identifiant normalisé : 30
Nombre de rejets étape 6 : 0


#Dans les données actuelles, tous les identifiants clients ont pu être corrigés au format `C000001`. Le nombre de rejets est donc égal à 0.

#### Étape 7 : nettoyage des noms

In [162]:
from pyspark.sql.functions import initcap, concat_ws

In [163]:
customers_step7 = (
    customers_step6
    .withColumn("first_name_clean", trim(regexp_replace(col("first_name"), "\\s+", " ")))
    .withColumn("first_name_clean",when(col("first_name_clean") == "", None).otherwise(initcap(col("first_name_clean"))))
    .withColumn("last_name_clean",trim(regexp_replace(col("last_name"), "\\s+", " ")))
    .withColumn("last_name_clean",when(col("last_name_clean") == "", None).otherwise(initcap(col("last_name_clean"))))
    .withColumn("full_name",concat_ws(" ", col("first_name_clean"), col("last_name_clean")))
)

In [164]:
customers_step7.select("customer_id","customer_id_clean","first_name","first_name_clean","last_name","last_name_clean","full_name").show(30, truncate=False)

+-----------+-----------------+----------+----------------+---------+---------------+--------------+
|customer_id|customer_id_clean|first_name|first_name_clean|last_name|last_name_clean|full_name     |
+-----------+-----------------+----------+----------------+---------+---------------+--------------+
|c001       |C000001          |Prenom1   |Prenom1         |Nom1     |Nom1           |Prenom1 Nom1  |
| C002      |C000002          |Prenom2   |Prenom2         |Nom2     |Nom2           |Prenom2 Nom2  |
|C003       |C000003          |Prenom3   |Prenom3         |Nom3     |Nom3           |Prenom3 Nom3  |
|c004       |C000004          |Prenom4   |Prenom4         |Nom4     |Nom4           |Prenom4 Nom4  |
|C005       |C000005          |Prenom5   |Prenom5         |Nom5     |Nom5           |Prenom5 Nom5  |
| C006      |C000006          |Prenom6   |Prenom6         |Nom6     |Nom6           |Prenom6 Nom6  |
| C007      |C000007          |Prenom7   |Prenom7         |Nom7     |Nom7           |Prenom

In [165]:
customers_step7.select(
    spark_sum(when(col("first_name_clean").isNull(), 1).otherwise(0)).alias("first_name_clean_nulls"),
    spark_sum(when(col("last_name_clean").isNull(), 1).otherwise(0)).alias("last_name_clean_nulls"),
    spark_sum(when(col("full_name").isNull(), 1).otherwise(0)).alias("full_name_nulls")
).show()

+----------------------+---------------------+---------------+
|first_name_clean_nulls|last_name_clean_nulls|full_name_nulls|
+----------------------+---------------------+---------------+
|                     0|                    0|              0|
+----------------------+---------------------+---------------+



#### Étape 8 : validation des adresses électroniques

In [166]:
from pyspark.sql.functions import lower, count

email_regex = r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$"

customers_step8 = (
    customers_step7
    .withColumn("email_tmp",lower(regexp_replace(trim(col("email")), "\\s+", "")))
    .withColumn("email_tmp",when(col("email_tmp") == "", None).otherwise(col("email_tmp")))
    .withColumn("is_email_valid",when(col("email_tmp").isNotNull() & col("email_tmp").rlike(email_regex),True).otherwise(False))
    .withColumn("email_clean",when(col("is_email_valid") == True, col("email_tmp")).otherwise(None)))

In [167]:
emails_dupliques = (
    customers_step8
    .filter(col("email_clean").isNotNull())
    .groupBy("email_clean")
    .agg(count("*").alias("nombre_occurrences"))
    .filter(col("nombre_occurrences") > 1))

emails_dupliques.show(truncate=False)

+-----------+------------------+
|email_clean|nombre_occurrences|
+-----------+------------------+
+-----------+------------------+



In [168]:
customers_step8 = (
    customers_step8
    .join(
        emails_dupliques
        .select("email_clean")
        .withColumn("is_email_duplicate", lit(True)),
        on="email_clean",
        how="left")
    .withColumn(
        "is_email_duplicate",
        when(col("is_email_duplicate").isNull(), False)
        .otherwise(col("is_email_duplicate"))))

In [169]:
customers_step8.select("customer_id_clean","full_name","email","email_tmp","email_clean","is_email_valid","is_email_duplicate").show(30, truncate=False)

+-----------------+--------------+------------------+------------------+------------------+--------------+------------------+
|customer_id_clean|full_name     |email             |email_tmp         |email_clean       |is_email_valid|is_email_duplicate|
+-----------------+--------------+------------------+------------------+------------------+--------------+------------------+
|C000010          |Prenom10 Nom10|user10@example.com|user10@example.com|user10@example.com|true          |false             |
|C000006          |Prenom6 Nom6  |user6@example.com |user6@example.com |user6@example.com |true          |false             |
|C000004          |Prenom4 Nom4  |user4@example.com |user4@example.com |user4@example.com |true          |false             |
|C000003          |Prenom3 Nom3  |user3@example.com |user3@example.com |user3@example.com |true          |false             |
|C000007          |Prenom7 Nom7  |user7example.com  |user7example.com  |NULL              |false         |false       

In [170]:
print("Nombre total de clients :", customers_step8.count())
print("Emails invalides :", customers_step8.filter(col("is_email_valid") == False).count())
print("Emails conservés dans email_clean :", customers_step8.filter(col("email_clean").isNotNull()).count())
print("Emails supprimés de email_clean :", customers_step8.filter(col("email_clean").isNull()).count())
print("Emails dupliqués :", customers_step8.filter(col("is_email_duplicate") == True).count())

Nombre total de clients : 30
Emails invalides : 4
Emails conservés dans email_clean : 26
Emails supprimés de email_clean : 4
Emails dupliqués : 0


#### Étape 9 : nettoyage des numéros de téléphone


In [171]:
from pyspark.sql.functions import substring

In [172]:
customers_step9 = (
    customers_step8
    .withColumn("phone_digits",regexp_replace(col("phone"), "[^0-9]", ""))
    .withColumn("phone_clean",when(
            col("phone_digits").rlike("^0[1-9][0-9]{8}$"),
            concat(lit("+33"), substring(col("phone_digits"), 2, 9)))
        .when(
            col("phone_digits").rlike("^33[1-9][0-9]{8}$"),
            concat(lit("+"), col("phone_digits")))
        .when(
            col("phone_digits").rlike("^0033[1-9][0-9]{8}$"),
            concat(lit("+33"), substring(col("phone_digits"), 5, 9)))
        .otherwise(None))
    .withColumn("is_phone_valid",when(col("phone_clean").isNotNull(), True).otherwise(False)))

In [173]:
customers_step9.select("customer_id_clean","full_name","phone","phone_digits","phone_clean","is_phone_valid").show(30, truncate=False)

+-----------------+--------------+-----------------+-------------+------------+--------------+
|customer_id_clean|full_name     |phone            |phone_digits |phone_clean |is_phone_valid|
+-----------------+--------------+-----------------+-------------+------------+--------------+
|C000001          |Prenom1 Nom1  |NULL             |NULL         |NULL        |false         |
|C000002          |Prenom2 Nom2  |0033612345678    |0033612345678|+33612345678|true          |
|C000003          |Prenom3 Nom3  |0612345678       |0612345678   |+33612345678|true          |
|C000004          |Prenom4 Nom4  |0612345678       |0612345678   |+33612345678|true          |
|C000005          |Prenom5 Nom5  |0033612345678    |0033612345678|+33612345678|true          |
|C000006          |Prenom6 Nom6  |+33 6 12 34 56 78|33612345678  |+33612345678|true          |
|C000007          |Prenom7 Nom7  |0033612345678    |0033612345678|+33612345678|true          |
|C000008          |Prenom8 Nom8  |NULL            

In [174]:
print("Nombre total de clients :", customers_step9.count())
print("Téléphones normalisés :", customers_step9.filter(col("phone_clean").isNotNull()).count())
print("Téléphones invalides ou absents :", customers_step9.filter(col("phone_clean").isNull()).count())

Nombre total de clients : 30
Téléphones normalisés : 25
Téléphones invalides ou absents : 5


In [175]:
customers_step9.filter(
    col("phone_clean").isNull()
).select("customer_id_clean","full_name","phone","phone_digits","phone_clean","is_phone_valid").show(truncate=False)

+-----------------+--------------+-----+------------+-----------+--------------+
|customer_id_clean|full_name     |phone|phone_digits|phone_clean|is_phone_valid|
+-----------------+--------------+-----+------------+-----------+--------------+
|C000001          |Prenom1 Nom1  |NULL |NULL        |NULL       |false         |
|C000008          |Prenom8 Nom8  |NULL |NULL        |NULL       |false         |
|C000015          |Prenom15 Nom15|NULL |NULL        |NULL       |false         |
|C000022          |Prenom22 Nom22|NULL |NULL        |NULL       |false         |
|C000026          |Prenom26 Nom26|NULL |NULL        |NULL       |false         |
+-----------------+--------------+-----+------------+-----------+--------------+



#### Étape 10 : normalisation des villes et des pays

In [176]:
from pyspark.sql.functions import lower, regexp_replace, trim

In [177]:
customers_step10_tmp = (
    customers_step9
    .withColumn("city_tmp",lower(trim(regexp_replace(col("city"), "\\s+", " "))))
    .withColumn("city_tmp",regexp_replace(col("city_tmp"), "-", " "))
    .withColumn("city_tmp",regexp_replace(col("city_tmp"), "\\s+", " "))
    .withColumn("city_tmp",trim(col("city_tmp")))
    .withColumn("country_tmp",lower(trim(regexp_replace(col("country"), "\\s+", " ")))))

In [178]:
customers_step10_tmp = (
    customers_step10_tmp
    .withColumn(
        "city_clean",
        when(col("city_tmp").rlike("^paris.*"), "Paris")
        .when(col("city_tmp").isin("lyon", "lyon 69000"), "Lyon")
        .when(col("city_tmp").isin("marseille", "marseille 13000"), "Marseille")
        .when(col("city_tmp").isin("toulouse"), "Toulouse")
        .when(col("city_tmp").isin("bordeaux"), "Bordeaux")
        .when(col("city_tmp").isin("lille"), "Lille")
        .when(col("city_tmp").isin("nantes"), "Nantes")
        .when(col("city_tmp").isin("nice"), "Nice")
        .otherwise(initcap(col("city_tmp")))))

In [179]:
customers_step10 = (
    customers_step10_tmp
    .withColumn(
        "country_clean",
        when(col("country_tmp").isin("fr", "france", "french republic"), "France")
        .when(col("country_tmp").isin("ma", "maroc", "morocco"), "Maroc")
        .when(col("country_tmp").isin("us", "usa", "united states"), "United States")
        .when(col("country_tmp").isin("uk", "gb", "united kingdom"), "United Kingdom")
        .otherwise(initcap(col("country_tmp")))))

In [180]:
customers_step10.select("customer_id_clean","full_name","city","city_tmp","city_clean","country","country_tmp","country_clean").show(30, truncate=False)

+-----------------+--------------+--------+--------+----------+---------------+---------------+-------------+
|customer_id_clean|full_name     |city    |city_tmp|city_clean|country        |country_tmp    |country_clean|
+-----------------+--------------+--------+--------+----------+---------------+---------------+-------------+
|C000001          |Prenom1 Nom1  |Paris   |paris   |Paris     |France         |france         |France       |
|C000002          |Prenom2 Nom2  |Paris   |paris   |Paris     |France         |france         |France       |
|C000003          |Prenom3 Nom3  |Lyon    |lyon    |Lyon      |French Republic|french republic|France       |
|C000004          |Prenom4 Nom4  |toulouse|toulouse|Toulouse  |france         |france         |France       |
|C000005          |Prenom5 Nom5  |PARIS   |paris   |Paris     |france         |france         |France       |
|C000006          |Prenom6 Nom6  |Paris 15|paris 15|Paris     |france         |france         |France       |
|C000007  

In [181]:
customers_step10.select(
    spark_sum(when(col("city_clean").isNull(), 1).otherwise(0)).alias("city_clean_nulls"),
    spark_sum(when(col("country_clean").isNull(), 1).otherwise(0)).alias("country_clean_nulls")
).show()

+----------------+-------------------+
|city_clean_nulls|country_clean_nulls|
+----------------+-------------------+
|               0|                  0|
+----------------+-------------------+



#### Étape 11 : conversion des dates de naissance

In [182]:
from pyspark.sql.functions import to_date, current_date, months_between, floor, coalesce

In [183]:
customers_step11_tmp = (
    customers_step10
    .withColumn(
        "birth_date_clean",
        coalesce(
            to_date(col("birth_date"), "yyyy-MM-dd"),
            to_date(col("birth_date"), "dd/MM/yyyy"),
            to_date(col("birth_date"), "yyyy/MM/dd"),
            to_date(col("birth_date"), "dd-MM-yyyy")))
    .withColumn("age",floor(months_between(current_date(), col("birth_date_clean")) / 12)))

In [184]:
customers_step11_tmp = (
    customers_step11_tmp
    .withColumn(
        "is_birth_date_valid",
        when(
            col("birth_date_clean").isNull()
            | (col("birth_date_clean") > current_date())
            | (col("age") < 0)
            | (col("age") > 120),
            False
        ).otherwise(True)))

In [185]:
customers_step11_tmp.select(
    "customer_id_clean",
    "full_name",
    "birth_date",
    "birth_date_clean",
    "age",
    "is_birth_date_valid"
).show(30, truncate=False)

+-----------------+--------------+----------+----------------+----+-------------------+
|customer_id_clean|full_name     |birth_date|birth_date_clean|age |is_birth_date_valid|
+-----------------+--------------+----------+----------------+----+-------------------+
|C000001          |Prenom1 Nom1  |1990/12/01|1990-12-01      |35  |true               |
|C000002          |Prenom2 Nom2  |1990-12-01|1990-12-01      |35  |true               |
|C000003          |Prenom3 Nom3  |1990/12/01|1990-12-01      |35  |true               |
|C000004          |Prenom4 Nom4  |NULL      |NULL            |NULL|false              |
|C000005          |Prenom5 Nom5  |1990-12-01|1990-12-01      |35  |true               |
|C000006          |Prenom6 Nom6  |31-02-2020|NULL            |NULL|false              |
|C000007          |Prenom7 Nom7  |NULL      |NULL            |NULL|false              |
|C000008          |Prenom8 Nom8  |1990/12/01|1990-12-01      |35  |true               |
|C000009          |Prenom9 Nom9 

In [186]:
customers_rejects_step11 = (
    customers_step11_tmp
    .filter(col("is_birth_date_valid") == False)
    .select(
        lit("customers").alias("source"),
        lit("Date de naissance invalide").alias("rejection_reason"),
        current_timestamp().alias("rejection_timestamp"),
        to_json(struct([col(c) for c in customers_step10.columns])).alias("original_data")))

customers_rejects_step11.show(truncate=False)

+---------+--------------------------+--------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|source   |rejection_reason          |rejection_timestamp       |original_data                                                                                                                                                                                                                     

In [187]:
customers_step11 = (
    customers_step11_tmp
    .filter(col("is_birth_date_valid") == True))

In [188]:
print("Nombre de lignes après étape 10 :", customers_step10.count())
print("Nombre de lignes après étape 11 :", customers_step11.count())
print("Nombre de rejets étape 11 :", customers_rejects_step11.count())

Nombre de lignes après étape 10 : 30
Nombre de lignes après étape 11 : 17
Nombre de rejets étape 11 : 13


#### Étape 12 : déduplication des clients

In [203]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

In [204]:
window_customers = Window.partitionBy(
    "customer_id_clean"
).orderBy(
    col("is_email_valid").desc(),
    col("created_at").desc())

In [205]:
customers_step12_tmp = (
    customers_step11_tmp
    .withColumn(
        "row_num",
        row_number().over(window_customers)))

In [206]:
customers_step12_tmp.select(
    "customer_id_clean",
    "row_num"
).show()

+-----------------+-------+
|customer_id_clean|row_num|
+-----------------+-------+
|          C000001|      1|
|          C000002|      1|
|          C000003|      1|
|          C000004|      1|
|          C000005|      1|
|          C000006|      1|
|          C000007|      1|
|          C000008|      1|
|          C000009|      1|
|          C000010|      1|
|          C000011|      1|
|          C000012|      1|
|          C000013|      1|
|          C000014|      1|
|          C000015|      1|
|          C000016|      1|
|          C000017|      1|
|          C000018|      1|
|          C000019|      1|
|          C000020|      1|
+-----------------+-------+
only showing top 20 rows



In [207]:
customers_clean = (
    customers_step12_tmp
    .filter(col("row_num") == 1)
    .drop("row_num"))

In [208]:
print("Avant déduplication :", customers_step11.count())
print("Après déduplication :", customers_clean.count())

Avant déduplication : 17
Après déduplication : 30


In [209]:
customers_clean.groupBy(
    "customer_id_clean"
).count().filter(
    col("count") > 1
).show()

+-----------------+-----+
|customer_id_clean|count|
+-----------------+-----+
+-----------------+-----+



Une colonne `row_num` a été créée avec `row_number()` afin d'identifier la meilleure ligne pour chaque client. 
Dans les données actuelles, aucun doublon n'a été détecté. Le nombre de clients reste donc inchangé après la déduplication

In [210]:
customers_clean.groupBy("email_clean").count().filter(
    col("email_clean").isNotNull()).filter(
    col("count") > 1).show(truncate=False)

+-----------+-----+
|email_clean|count|
+-----------+-----+
+-----------+-----+



In [211]:
customers_clean.groupBy("phone_clean").count().filter(
    col("phone_clean").isNotNull()).filter(col("count") > 1).show(truncate=False)

+------------+-----+
|phone_clean |count|
+------------+-----+
|+33612345678|25   |
+------------+-----+



In [212]:
customers_clean.groupBy(
    "first_name_clean",
    "last_name_clean",
    "birth_date_clean"
).count().filter(
    col("count") > 1
).show(truncate=False)


+----------------+---------------+----------------+-----+
|first_name_clean|last_name_clean|birth_date_clean|count|
+----------------+---------------+----------------+-----+
+----------------+---------------+----------------+-----+



In [213]:
customers_clean.filter(
    col("phone_clean") == "+33612345678"
).select(
    "customer_id_clean",
    "full_name",
    "email_clean",
    "phone_clean"
).show(20, truncate=False)

+-----------------+--------------+------------------+------------+
|customer_id_clean|full_name     |email_clean       |phone_clean |
+-----------------+--------------+------------------+------------+
|C000002          |Prenom2 Nom2  |user2@example.com |+33612345678|
|C000003          |Prenom3 Nom3  |user3@example.com |+33612345678|
|C000004          |Prenom4 Nom4  |user4@example.com |+33612345678|
|C000005          |Prenom5 Nom5  |user5@example.com |+33612345678|
|C000006          |Prenom6 Nom6  |user6@example.com |+33612345678|
|C000007          |Prenom7 Nom7  |NULL              |+33612345678|
|C000009          |Prenom9 Nom9  |user9@example.com |+33612345678|
|C000010          |Prenom10 Nom10|user10@example.com|+33612345678|
|C000011          |Prenom11 Nom11|user11@example.com|+33612345678|
|C000012          |Prenom12 Nom12|user12@example.com|+33612345678|
|C000013          |Prenom13 Nom13|user13@example.com|+33612345678|
|C000014          |Prenom14 Nom14|NULL              |+33612345

Aucun doublon n'a été détecté selon :
- customer_id_clean ;
- email_clean ;
- la combinaison nom, prénom et date de naissance.

En revanche, plusieurs clients partagent le numéro de téléphone +33612345678.

Cette situation a été identifiée comme une anomalie de qualité des données. Elle ne permet toutefois pas d'affirmer qu'il s'agit de clients dupliqués, car les autres attributs des clients sont différents.

## Partie 5 — Nettoyage des commandes

#### Étape 13 : normalisation des commandes

In [214]:
from pyspark.sql.functions import to_timestamp

In [215]:
orders_step13_tmp = (
    orders_df
    .withColumn("order_id_tmp", upper(trim(col("order_id"))))
    .withColumn("order_id_tmp",regexp_replace(col("order_id_tmp"), "[-_\\s]", ""))
    .withColumn("order_id_number",regexp_extract(col("order_id_tmp"),"^ORD0*([0-9]+)$",1))
    .withColumn("order_id_clean",when(col("order_id_number") != "",concat(lit("ORD"),lpad(col("order_id_number"), 6, "0")))
    .otherwise(None)))

In [216]:
orders_step13_tmp = (
    orders_step13_tmp
    .withColumn("customer_id_step1",upper(trim(col("customer_id"))))
    .withColumn("customer_id_step2",regexp_replace(col("customer_id_step1"),"[-_\\s]",""))
    .withColumn("customer_id_step3",regexp_replace(col("customer_id_step2"),"^CUST","C"))
    .withColumn("customer_id_number",regexp_extract(col("customer_id_step3"),"^C0*([0-9]+)$",1))
    .withColumn("customer_id_clean",when(col("customer_id_number") != "",concat(lit("C"),lpad(col("customer_id_number"), 6, "0")))
    .otherwise(None)))

In [217]:
orders_step13_tmp.select(
    "order_id",
    "order_id_clean",
    "customer_id",
    "customer_id_clean"
).show(20, truncate=False)

+--------+--------------+-----------+-----------------+
|order_id|order_id_clean|customer_id|customer_id_clean|
+--------+--------------+-----------+-----------------+
|ord-0001|ORD000001     |C999       |C000999          |
| ORD2   |ORD000002     |c004       |C000004          |
| ORD3   |ORD000003     |cust-26    |C000026          |
|ord-0004|ORD000004     |c004       |C000004          |
|ORD-0005|ORD000005     | C002      |C000002          |
| ORD6   |ORD000006     |cust-26    |C000026          |
| ORD7   |ORD000007     | C002      |C000002          |
|ord-0008|ORD000008     | C019      |C000019          |
| ORD9   |ORD000009     |c022       |C000022          |
| ORD10  |ORD000010     |C016       |C000016          |
|ord-0011|ORD000011     | C007      |C000007          |
| ORD12  |ORD000012     |c001       |C000001          |
| ORD13  |ORD000013     |c022       |C000022          |
|ord-0014|ORD000014     |C009       |C000009          |
|ord-0015|ORD000015     |cust-27    |C000027    

In [218]:
orders_step13_tmp = (
    orders_step13_tmp
    .withColumn(
        "order_date_clean",
        coalesce(
            to_timestamp(col("order_date"), "yyyy-MM-dd HH:mm:ss"),
            to_timestamp(col("order_date"), "yyyy-MM-dd"),
            to_timestamp(col("order_date"), "dd/MM/yyyy"),
            to_timestamp(col("order_date"), "yyyy/MM/dd"),
            to_timestamp(col("order_date"), "dd-MM-yyyy"))))

In [219]:
orders_step13_tmp.select(
    "order_id",
    "order_date",
    "order_date_clean"
).show(20, truncate=False)

+--------+----------+-------------------+
|order_id|order_date|order_date_clean   |
+--------+----------+-------------------+
|ord-0001|2026-03-01|2026-03-01 00:00:00|
| ORD2   |2026-03-01|2026-03-01 00:00:00|
| ORD3   |01/03/2026|2026-03-01 00:00:00|
|ord-0004|2026/03/01|2026-03-01 00:00:00|
|ORD-0005|2026-03-01|2026-03-01 00:00:00|
| ORD6   |2026/03/01|2026-03-01 00:00:00|
| ORD7   |2026/03/01|2026-03-01 00:00:00|
|ord-0008|bad-date  |NULL               |
| ORD9   |bad-date  |NULL               |
| ORD10  |bad-date  |NULL               |
|ord-0011|2026-03-01|2026-03-01 00:00:00|
| ORD12  |bad-date  |NULL               |
| ORD13  |2026/03/01|2026-03-01 00:00:00|
|ord-0014|2026/03/01|2026-03-01 00:00:00|
|ord-0015|2026-03-01|2026-03-01 00:00:00|
|ord-0016|2026/03/01|2026-03-01 00:00:00|
|ord-0017|2026/03/01|2026-03-01 00:00:00|
|ORD-0018|bad-date  |NULL               |
| ORD19  |2026/03/01|2026-03-01 00:00:00|
| ORD20  |2026/03/01|2026-03-01 00:00:00|
+--------+----------+-------------

In [220]:
orders_rejects_step13 = (
    orders_step13_tmp
    .filter(col("order_date_clean").isNull())
    .select(
        lit("orders").alias("source"),
        lit("Date de commande invalide").alias("rejection_reason"),
        current_timestamp().alias("rejection_timestamp"),
        to_json(struct([col(c) for c in orders_df.columns])).alias("original_data")))

In [221]:
print("Nombre de rejets étape 13 :", orders_rejects_step13.count())
orders_rejects_step13.show(truncate=False)

Nombre de rejets étape 13 : 7
+------+-------------------------+-------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|source|rejection_reason         |rejection_timestamp      |original_data                                                                                                                                                           |
+------+-------------------------+-------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|orders|Date de commande invalide|2026-07-24 07:59:35.11101|{"order_id":"ord-0008","customer_id":" C019","order_date":"bad-date","status":"annulée","payment_method":"CARD","total_amount":-20.000000000000000000}                  |
|orders|Date de commande invalide|2026-07-24 07:59

In [222]:
orders_clean_step13 = (
    orders_step13_tmp
    .filter(col("order_date_clean").isNotNull()))

In [223]:
print("Nombre de commandes initial :", orders_df.count())
print("Nombre de commandes conservées :", orders_clean_step13.count())

Nombre de commandes initial : 40
Nombre de commandes conservées : 33


#### Étape 14 : normalisation des statuts

In [224]:
orders_clean_step13.select("status").distinct().show(truncate=False)

+---------+
|status   |
+---------+
|completed|
|cancelled|
|PAID     |
|paid     |
|annulée  |
|Payée    |
+---------+



In [225]:
orders_step14 = (
    orders_clean_step13
    .withColumn("status_tmp",upper(trim(col("status"))))
    .withColumn(
        "order_status",
        when(col("status_tmp").isin("PAID", "PAYÉE"), "PAID")
        .when(col("status_tmp").isin("COMPLETED"), "DELIVERED")
        .when(col("status_tmp").isin("CANCELLED", "ANNULÉE"), "CANCELLED")
        .when(col("status_tmp").isin("CREATED"), "CREATED")
        .when(col("status_tmp").isin("PREPARING"), "PREPARING")
        .when(col("status_tmp").isin("SHIPPED"), "SHIPPED")
        .when(col("status_tmp").isin("RETURNED"), "RETURNED")
        .otherwise("UNKNOWN")))

In [226]:
orders_step14.select(
    "status",
    "status_tmp",
    "order_status"
).distinct().show(truncate=False)

+---------+----------+------------+
|status   |status_tmp|order_status|
+---------+----------+------------+
|completed|COMPLETED |DELIVERED   |
|Payée    |PAYÉE     |PAID        |
|PAID     |PAID      |PAID        |
|cancelled|CANCELLED |CANCELLED   |
|annulée  |ANNULÉE   |CANCELLED   |
|paid     |PAID      |PAID        |
+---------+----------+------------+



In [227]:
orders_step14.groupBy(
    "order_status"
).count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|   DELIVERED|    4|
|        PAID|   17|
|   CANCELLED|   12|
+------------+-----+



Les statuts de commande ont été harmonisés dans la colonne order_status.

Les variantes textuelles ont été regroupées dans des catégories standardisées :
- paid, PAID et Payée = PAID
- completed = DELIVERED
- cancelled et annulée = CANCELLED

Les statuts non reconnus sont remplacés par UNKNOWN.

#### Étape 15 : traitement des montants et devises

In [228]:
orders_step14.select(
    "currency"
).distinct().show(truncate=False)

+--------+
|currency|
+--------+
|EUR     |
|MAD     |
|USD     |
|NULL    |
+--------+



In [229]:
orders_step14.select(
    "total_amount"
).show(20, truncate=False)

+----------------------+
|total_amount          |
+----------------------+
|100.500000000000000000|
|-20.000000000000000000|
|100.500000000000000000|
|100.500000000000000000|
|-20.000000000000000000|
|-20.000000000000000000|
|55.900000000000000000 |
|100.500000000000000000|
|100.500000000000000000|
|100.500000000000000000|
|55.900000000000000000 |
|100.500000000000000000|
|-20.000000000000000000|
|-20.000000000000000000|
|100.500000000000000000|
|55.900000000000000000 |
|-20.000000000000000000|
|55.900000000000000000 |
|-20.000000000000000000|
|100.500000000000000000|
+----------------------+
only showing top 20 rows



In [230]:
# Montants invalides

orders_negative_amounts = (
    orders_step14
    .filter(col("total_amount") <= 0))

print(
    "Montants négatifs ou nuls :",
    orders_negative_amounts.count())

orders_negative_amounts.select(
    "order_id",
    "total_amount"
).show(truncate=False)

Montants négatifs ou nuls : 12
+--------+----------------------+
|order_id|total_amount          |
+--------+----------------------+
| ORD2   |-20.000000000000000000|
|ORD-0005|-20.000000000000000000|
| ORD6   |-20.000000000000000000|
|ord-0017|-20.000000000000000000|
| ORD19  |-20.000000000000000000|
|ORD-0023|-20.000000000000000000|
| ORD25  |-20.000000000000000000|
|ORD-0031|-20.000000000000000000|
|ORD-0032|-20.000000000000000000|
| ORD33  |-20.000000000000000000|
|ord-0037|-20.000000000000000000|
|ord-0040|-20.000000000000000000|
+--------+----------------------+



In [231]:
# Devises invalides
allowed_currencies = [
    "EUR",
    "USD",
    "GBP",
    "MAD"
]

orders_invalid_currency = (
    orders_step14
    .filter(col("currency").isNull()|(~col("currency").isin(allowed_currencies))))

print("Devises invalides ou absentes :",orders_invalid_currency.count())

orders_invalid_currency.select(
    "order_id",
    "currency"
).show(truncate=False)


Devises invalides ou absentes : 11
+--------+--------+
|order_id|currency|
+--------+--------+
| ORD2   |NULL    |
| ORD3   |NULL    |
|ord-0016|NULL    |
| ORD19  |NULL    |
| ORD21  |NULL    |
| ORD27  |NULL    |
|ord-0030|NULL    |
|ORD-0032|NULL    |
| ORD33  |NULL    |
|ord-0034|NULL    |
|ORD-0035|NULL    |
+--------+--------+



In [232]:
# DataFrame des taux de change
exchange_rates = spark.createDataFrame(
    [
        ("EUR", 1.00),
        ("USD", 0.92),
        ("GBP", 1.17),
        ("MAD", 0.092)
    ],
    [
        "currency",
        "rate_to_eur"
    ])

exchange_rates.show()

+--------+-----------+
|currency|rate_to_eur|
+--------+-----------+
|     EUR|        1.0|
|     USD|       0.92|
|     GBP|       1.17|
|     MAD|      0.092|
+--------+-----------+



In [233]:
from pyspark.sql.functions import broadcast, round
orders_step15 = (orders_step14.join(broadcast(exchange_rates),on="currency",how="left"))

In [234]:
orders_step15 = (orders_step15.withColumn("total_amount_eur",round(col("total_amount") * col("rate_to_eur"),2)))

In [235]:
orders_step15.select(
    "order_id",
    "currency",
    "total_amount",
    "rate_to_eur",
    "total_amount_eur"
).show(20, truncate=False)

+--------+--------+----------------------+-----------+----------------+
|order_id|currency|total_amount          |rate_to_eur|total_amount_eur|
+--------+--------+----------------------+-----------+----------------+
|ord-0001|MAD     |100.500000000000000000|0.092      |9.25            |
| ORD2   |NULL    |-20.000000000000000000|NULL       |NULL            |
| ORD3   |NULL    |100.500000000000000000|NULL       |NULL            |
|ord-0004|EUR     |100.500000000000000000|1.0        |100.5           |
|ORD-0005|USD     |-20.000000000000000000|0.92       |-18.4           |
| ORD6   |EUR     |-20.000000000000000000|1.0        |-20.0           |
| ORD7   |USD     |55.900000000000000000 |0.92       |51.43           |
|ord-0011|EUR     |100.500000000000000000|1.0        |100.5           |
| ORD13  |MAD     |100.500000000000000000|0.092      |9.25            |
|ord-0014|EUR     |100.500000000000000000|1.0        |100.5           |
|ord-0015|EUR     |55.900000000000000000 |1.0        |55.9      

In [236]:
orders_rejects_step15 = (
    orders_step15
    .filter((col("total_amount") <= 0)|col("currency").isNull()|(~col("currency").isin(allowed_currencies))))

In [237]:
print("Montants négatifs :", orders_negative_amounts.count())

print("Devises invalides ou absentes :",
      orders_invalid_currency.count())

print("Nombre total de rejets étape 15 :",
      orders_rejects_step15.count())

Montants négatifs : 12
Devises invalides ou absentes : 11
Nombre total de rejets étape 15 : 19


In [238]:
orders_clean = (
    orders_step15
    .filter(col("total_amount") > 0)
    .filter(col("currency").isin(allowed_currencies)))

print("Nombre initial :", orders_step15.count())
print("Nombre conservé :", orders_clean.count())

Nombre initial : 33
Nombre conservé : 14


## Partie 6 — Nettoyage des lignes de commande

#### Étape 16 : validation des quantités et des prix

In [239]:
order_items_df.show(20, truncate=False)

+--------+----------+--------+----------------------+--------------------+
|order_id|product_id|quantity|unit_price            |discount            |
+--------+----------+--------+----------------------+--------------------+
|ord-0036|P012      |-1      |100.000000000000000000|0.100000000000000000|
|ord-0028|P001      |-1      |10.000000000000000000 |0.100000000000000000|
|ORD-0031|P002      |1       |25.000000000000000000 |0.200000000000000000|
| ORD21  |P010      |2       |10.000000000000000000 |0.000000000000000000|
|ord-0040|P012      |-1      |100.000000000000000000|1.200000000000000000|
| ORD13  |P005      |2       |25.000000000000000000 |0.200000000000000000|
|ord-0008|P015      |0       |10.000000000000000000 |0.200000000000000000|
| ORD9   |P014      |2       |-5.000000000000000000 |NULL                |
| ORD7   |P013      |3       |100.000000000000000000|0.200000000000000000|
| ORD10  |P012      |3       |-5.000000000000000000 |0.100000000000000000|
|ord-0004|P001      |3   

In [240]:
order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(38,18) (nullable = true)
 |-- discount: decimal(38,18) (nullable = true)



In [241]:
order_items_step16_tmp = (
    order_items_df
    .withColumn(
        "discount",
        when(col("discount").isNull(), 0)
        .otherwise(col("discount"))))

In [242]:
order_items_step16_tmp = (
    order_items_step16_tmp
    .withColumn(
        "is_valid",
        when(
            (col("quantity") <= 0)
            | (col("unit_price") < 0)
            | (col("discount") < 0)
            | (col("discount") > 1),
            False)
        .otherwise(True)))


In [243]:
order_items_rejects_step16 = (
    order_items_step16_tmp
    .filter(col("is_valid") == False)
    .select(
        lit("order_items").alias("source"),
        lit("Quantité, prix ou remise invalide").alias("rejection_reason"),
        current_timestamp().alias("rejection_timestamp"),
        to_json(
            struct([col(c) for c in order_items_df.columns])
        ).alias("original_data")))

In [244]:
print(
    "Nombre de rejets étape 16 :",
    order_items_rejects_step16.count())

order_items_rejects_step16.show(truncate=False)

Nombre de rejets étape 16 : 56
+-----------+---------------------------------+--------------------------+-----------------------------------------------------------------------------------------------------------------------------+
|source     |rejection_reason                 |rejection_timestamp       |original_data                                                                                                                |
+-----------+---------------------------------+--------------------------+-----------------------------------------------------------------------------------------------------------------------------+
|order_items|Quantité, prix ou remise invalide|2026-07-24 07:59:59.203389|{"order_id":"ord-0036","product_id":"P012","quantity":-1,"unit_price":100.000000000000000000,"discount":0.100000000000000000}|
|order_items|Quantité, prix ou remise invalide|2026-07-24 07:59:59.203389|{"order_id":"ord-0028","product_id":"P001","quantity":-1,"unit_price":10.000000000000000000

In [245]:
order_items_step16 = (
    order_items_step16_tmp
    .filter(col("is_valid") == True))

In [246]:
# Calcul des montants 

order_items_step16 = (
    order_items_step16
    .withColumn("gross_amount",col("quantity") * col("unit_price"))
    .withColumn("discount_amount",col("gross_amount") * col("discount"))
    .withColumn("net_amount",col("gross_amount") - col("discount_amount")))

In [247]:
order_items_step16.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "gross_amount",
    "discount_amount",
    "net_amount"
).show(30, truncate=False)

+--------+----------+--------+----------------------+--------------------+------------+---------------+----------+
|order_id|product_id|quantity|unit_price            |discount            |gross_amount|discount_amount|net_amount|
+--------+----------+--------+----------------------+--------------------+------------+---------------+----------+
|ORD-0031|P002      |1       |25.000000000000000000 |0.200000000000000000|25.0000000  |5.000000       |20.000000 |
| ORD21  |P010      |2       |10.000000000000000000 |0.000000000000000000|20.0000000  |0.000000       |20.000000 |
| ORD13  |P005      |2       |25.000000000000000000 |0.200000000000000000|50.0000000  |10.000000      |40.000000 |
| ORD7   |P013      |3       |100.000000000000000000|0.200000000000000000|300.0000000 |60.000000      |240.000000|
|ord-0004|P009      |2       |25.000000000000000000 |0.100000000000000000|50.0000000  |5.000000       |45.000000 |
| ORD21  |P010      |1       |10.000000000000000000 |0.000000000000000000|10.000

In [248]:
from pyspark.sql.functions import (
    countDistinct,
    sum as spark_sum)

order_items_step16 = (
    order_items_step16
    .withColumn(
        "order_id_tmp",
        upper(trim(col("order_id"))))
    .withColumn(
        "order_id_tmp",
        regexp_replace(col("order_id_tmp"), "[-_\\s]", ""))
    .withColumn(
        "order_id_number",
        regexp_extract(
            col("order_id_tmp"),
            "^ORD0*([0-9]+)$",1))
    .withColumn(
        "order_id_clean",
        when(
            col("order_id_number") != "",
            concat(
                lit("ORD"),
                lpad(col("order_id_number"), 6, "0"))
        ).otherwise(None)))

In [249]:
# agrégation par commande

order_items_summary = (
    order_items_step16
    .groupBy("order_id_clean")
    .agg(
        countDistinct("product_id").alias("number_of_products"),
        spark_sum("quantity").alias("total_quantity"),
        spark_sum("gross_amount").alias("gross_amount_total"),
        spark_sum("discount_amount").alias("discount_total"),
        spark_sum("net_amount").alias("net_amount_total")))

In [250]:
order_items_summary.show(30, truncate=False)

+--------------+------------------+--------------+------------------+--------------+----------------+
|order_id_clean|number_of_products|total_quantity|gross_amount_total|discount_total|net_amount_total|
+--------------+------------------+--------------+------------------+--------------+----------------+
|ORD000027     |1                 |3             |300.0000000       |0.000000      |300.000000      |
|ORD000036     |1                 |1             |25.0000000        |0.000000      |25.000000       |
|ORD000017     |1                 |1             |100.0000000       |0.000000      |100.000000      |
|ORD000009     |1                 |3             |75.0000000        |0.000000      |75.000000       |
|ORD000032     |1                 |2             |200.0000000       |20.000000     |180.000000      |
|ORD000004     |1                 |2             |50.0000000        |5.000000      |45.000000       |
|ORD000031     |3                 |6             |375.0000000       |5.000000     

#### Étape 17 : comparaison des montants

In [251]:
amount_comparison = (
    orders_clean
    .join(
        order_items_summary,
        on="order_id_clean",
        how="left"))

In [252]:
from pyspark.sql.functions import abs

amount_comparison = (
    amount_comparison
    .withColumn(
        "amount_difference",
        abs(col("total_amount_eur") - col("net_amount_total")))
    .withColumn(
        "is_amount_consistent",
        when(
            col("amount_difference") <= 0.01,
            True
        ).otherwise(False)))

In [253]:
amount_comparison.select(
    "order_id_clean",
    "total_amount_eur",
    "net_amount_total",
    "amount_difference",
    "is_amount_consistent"
).show(30, truncate=False)

+--------------+----------------+----------------+-----------------+--------------------+
|order_id_clean|total_amount_eur|net_amount_total|amount_difference|is_amount_consistent|
+--------------+----------------+----------------+-----------------+--------------------+
|ORD000024     |5.14            |NULL            |NULL             |false               |
|ORD000036     |9.25            |25.000000       |15.75            |false               |
|ORD000015     |55.9            |NULL            |NULL             |false               |
|ORD000028     |55.9            |NULL            |NULL             |false               |
|ORD000004     |100.5           |45.000000       |55.5             |false               |
|ORD000007     |51.43           |285.000000      |233.57           |false               |
|ORD000026     |100.5           |NULL            |NULL             |false               |
|ORD000014     |100.5           |NULL            |NULL             |false               |
|ORD000020

## Partie 7 — Nettoyage des produits

#### Étape 18 : normalisation du catalogue

In [254]:
products_df.show(20, truncate=False)

+----------+------------+------------+------+----------------------+------+
|product_id|product_name|category    |brand |current_price         |active|
+----------+------------+------------+------+----------------------+------+
|P001      |Produit 1   |Electronique|HP    |211.000000000000000000|true  |
|P002      |Produit 2   |High-Tech   |Dell  |42.000000000000000000 |true  |
|P003      |Produit 3   |High-Tech   |HP    |485.000000000000000000|true  |
|P004      |Produit 4   |informatique|Lenovo|101.000000000000000000|true  |
|P005      |Produit 5   |informatique|HP    |265.000000000000000000|true  |
|P006      |Produit 6   |high tech   |HP    |83.000000000000000000 |false |
|P007      |Produit 7   |informatique|HP    |175.000000000000000000|false |
|P008      |Produit 8   |Electronique|Lenovo|29.000000000000000000 |true  |
|P009      |Produit 9   |informatique|Lenovo|335.000000000000000000|true  |
|P010      |Produit 10  |High-Tech   |Dell  |208.000000000000000000|false |
|P011      |

In [255]:
products_df.select("category").distinct().show(truncate=False)
products_df.select("brand").distinct().show(truncate=False)

+------------+
|category    |
+------------+
|High-Tech   |
|INFORMATIQUE|
|high tech   |
|informatique|
|Electronique|
+------------+

+------+
|brand |
+------+
|HP    |
|Dell  |
|Lenovo|
+------+



In [256]:
# Normalisation de l'identifiant produit

products_step18_tmp = (
    products_df
    .withColumn("product_id_tmp",upper(trim(col("product_id"))))
    .withColumn("product_id_tmp",regexp_replace(col("product_id_tmp"),"[-_\\s]",""))
    .withColumn("product_id_clean",col("product_id_tmp")))

In [257]:
#Nettoyage du nom produit

products_step18_tmp = (
    products_step18_tmp
    .withColumn("product_name_clean",
    when(trim(col("product_name")) == "",None)
    .otherwise(initcap(trim(regexp_replace(col("product_name"),"\\s+"," "))))))

In [258]:
# Nettoyage de la marque

products_step18_tmp = (
    products_step18_tmp
    .withColumn(
        "brand_clean",
        when(trim(col("brand")) == "",None)
        .otherwise(initcap(trim(regexp_replace(col("brand"),"\\s+"," "))))))

In [259]:
#Normalisation des catégories

products_step18 = (
    products_step18_tmp
    .withColumn("category_tmp",lower(trim(regexp_replace(col("category"),"\\s+"," "))))
    .withColumn(
        "category_clean",
        when(col("category_tmp").isin(
                "high-tech",
                "high tech",
                "informatique",
                "electronique",
                "électronique"),"Technologie")
        .otherwise(initcap(col("category_tmp")))))

In [260]:
products_step18.select(
    "product_id",
    "product_id_clean",
    "product_name",
    "product_name_clean",
    "category",
    "category_clean",
    "brand",
    "brand_clean"
).show(30, truncate=False)

+----------+----------------+------------+------------------+------------+--------------+------+-----------+
|product_id|product_id_clean|product_name|product_name_clean|category    |category_clean|brand |brand_clean|
+----------+----------------+------------+------------------+------------+--------------+------+-----------+
|P001      |P001            |Produit 1   |Produit 1         |Electronique|Technologie   |HP    |Hp         |
|P002      |P002            |Produit 2   |Produit 2         |High-Tech   |Technologie   |Dell  |Dell       |
|P003      |P003            |Produit 3   |Produit 3         |High-Tech   |Technologie   |HP    |Hp         |
|P004      |P004            |Produit 4   |Produit 4         |informatique|Technologie   |Lenovo|Lenovo     |
|P005      |P005            |Produit 5   |Produit 5         |informatique|Technologie   |HP    |Hp         |
|P006      |P006            |Produit 6   |Produit 6         |high tech   |Technologie   |HP    |Hp         |
|P007      |P007   

In [261]:
# Déduplication des produits

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [262]:
window_products = Window.partitionBy("product_id_clean").orderBy(col("active").desc())

In [263]:
products_step18 = (products_step18.withColumn("row_num",row_number().over(window_products)))

In [264]:
products_clean = (products_step18.filter(col("row_num") == 1).drop("row_num"))

In [265]:
print("Nombre initial :", products_df.count())
print("Nombre final :", products_clean.count())

Nombre initial : 15
Nombre final : 15


In [266]:
products_clean.groupBy("product_id_clean").count().filter(col("count") > 1).show()

+----------------+-----+
|product_id_clean|count|
+----------------+-----+
+----------------+-----+



## Partie 8 — Nettoyage des avis MongoDB

#### Étape 19 : normalisation des avis

In [267]:
reviews_df.printSchema()

root
 |-- _id: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- customerId: string (nullable = true)
 |-- orderId: string (nullable = true)
 |-- productId: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- reviewDate: string (nullable = true)
 |-- verifiedPurchase: string (nullable = true)



In [268]:
reviews_df.show(20, truncate=False)

+------+----------------+----------+--------+---------+------+-------------------+----------------+
|_id   |comment         |customerId|orderId |productId|rating|reviewDate         |verifiedPurchase|
+------+----------------+----------+--------+---------+------+-------------------+----------------+
|REV001|Très bon produit|cust-26   | ORD33  |P009     |7     |2026-03-12T10:25:00|yes             |
|REV002|Très bon produit| C019     | ORD33  |P013     |3     |2026-03-12T10:25:00|yes             |
|REV003|Très bon produit| C018     | ORD27  |P004     |5     |2026-03-12T10:25:00|0               |
|REV004|Très bon produit|c001      |ord-0028|P011     |5     |2026-03-12T10:25:00|false           |
|REV005|                |c022      | ORD25  |P002     |7     |2026-03-12T10:25:00|1               |
|REV006|Très bon produit|C010      | ORD2   |P011     |7     |2026-03-12T10:25:00|no              |
|REV007|Très bon produit|c022      | ORD9   |P002     |2     |2026-03-12T10:25:00|false           |


In [269]:
reviews_df.select("verifiedPurchase").distinct().show(truncate=False)

+----------------+
|verifiedPurchase|
+----------------+
|0               |
|false           |
|1               |
|no              |
|yes             |
|true            |
+----------------+



In [270]:
# Normalisation du customerId

reviews_step19 = (
    reviews_df.withColumn("customer_id_tmp",upper(trim(col("customerId"))))
    .withColumn("customer_id_tmp",regexp_replace(col("customer_id_tmp"),"[-_\\s]", ""))
    .withColumn("customer_id_tmp",regexp_replace(col("customer_id_tmp"),"^CUST","C"))
    .withColumn("customer_id_number",regexp_extract(col("customer_id_tmp"),"^C0*([0-9]+)$",1))
    .withColumn("customer_id_clean",
        when(col("customer_id_number") != "",concat(lit("C"),lpad(col("customer_id_number"), 6, "0"))).otherwise(None)))

In [271]:
# Normalisation du productId

reviews_step19 = (reviews_step19.withColumn("product_id_clean",upper(regexp_replace(trim(col("productId")),"[-_\\s]",""))))

In [272]:
# Normalisation du orderId

reviews_step19 = (
    reviews_step19
    .withColumn("order_id_tmp",upper(trim(col("orderId"))))
    .withColumn("order_id_tmp",regexp_replace(col("order_id_tmp"),"[-_\\s]",""))
    .withColumn("order_id_number",regexp_extract(col("order_id_tmp"),"^ORD0*([0-9]+)$",1))
    .withColumn("order_id_clean",when(col("order_id_number") != "",concat(lit("ORD"),lpad(col("order_id_number"), 6, "0")))
    .otherwise(None)))

In [273]:
# Conversion de reviewDate

reviews_step19 = (
    reviews_step19
    .withColumn("review_date",coalesce(to_timestamp(col("reviewDate")),to_timestamp(col("reviewDate"),"yyyy-MM-dd HH:mm:ss"))))

In [274]:
reviews_step19 = (
    reviews_step19
    .withColumn("verified_purchase",
        when(lower(col("verifiedPurchase").cast("string")).isin("true","1","yes"),True)
        .when(lower(col("verifiedPurchase").cast("string")).isin("false","0","no"),False)
        .otherwise(None)))

In [275]:
reviews_step19.select(
    "customerId",
    "customer_id_clean",
    "productId",
    "product_id_clean",
    "orderId",
    "order_id_clean",
    "reviewDate",
    "review_date",
    "verifiedPurchase",
    "verified_purchase"
).show(30, truncate=False)

+----------+-----------------+---------+----------------+--------+--------------+-------------------+-------------------+----------------+-----------------+
|customerId|customer_id_clean|productId|product_id_clean|orderId |order_id_clean|reviewDate         |review_date        |verifiedPurchase|verified_purchase|
+----------+-----------------+---------+----------------+--------+--------------+-------------------+-------------------+----------------+-----------------+
|cust-26   |C000026          |P009     |P009            | ORD33  |ORD000033     |2026-03-12T10:25:00|2026-03-12 10:25:00|yes             |true             |
| C019     |C000019          |P013     |P013            | ORD33  |ORD000033     |2026-03-12T10:25:00|2026-03-12 10:25:00|yes             |true             |
| C018     |C000018          |P004     |P004            | ORD27  |ORD000027     |2026-03-12T10:25:00|2026-03-12 10:25:00|0               |false            |
|c001      |C000001          |P011     |P011            |o

In [276]:
reviews_step19.select("verifiedPurchase","verified_purchase").show(30, truncate=False)

+----------------+-----------------+
|verifiedPurchase|verified_purchase|
+----------------+-----------------+
|yes             |true             |
|yes             |true             |
|0               |false            |
|false           |false            |
|1               |true             |
|no              |false            |
|false           |false            |
|false           |false            |
|true            |true             |
|false           |false            |
|true            |true             |
|false           |false            |
|0               |false            |
|yes             |true             |
|true            |true             |
|0               |false            |
|0               |false            |
|true            |true             |
|true            |true             |
|false           |false            |
|no              |false            |
|0               |false            |
|0               |false            |
|false           |false            |
|

In [277]:
reviews_step19.groupBy("verified_purchase").count().show()

+-----------------+-----+
|verified_purchase|count|
+-----------------+-----+
|             true|   12|
|            false|   18|
+-----------------+-----+



#### Étape 20 : validation des notes

In [278]:
reviews_step19.select("rating").distinct().show()

+------+
|rating|
+------+
|     1|
|     3|
|     5|
|     4|
|     7|
|     2|
+------+



In [279]:
# Validation des notes

reviews_step20 = (
    reviews_step19
    .withColumn("is_rating_valid",when((col("rating") >= 1) & (col("rating") <= 5),True).otherwise(False)))

In [280]:
# Nettoyage des commentaires

reviews_step20 = (
    reviews_step20
    .withColumn("comment",when(trim(col("comment")) == "",None).otherwise(col("comment"))))

In [281]:
reviews_rejects_step20 = (
    reviews_step20
    .filter(col("is_rating_valid") == False)
    .select(
        lit("reviews").alias("source"),
        lit("Note hors intervalle [1,5]").alias("rejection_reason"),
        current_timestamp().alias("rejection_timestamp"),
        to_json(struct([col(c) for c in reviews_df.columns])
        ).alias("original_data")))


In [282]:
reviews_step20.groupBy("is_rating_valid").count().show()

+---------------+-----+
|is_rating_valid|count|
+---------------+-----+
|           true|   20|
|          false|   10|
+---------------+-----+



In [283]:
reviews_rejects_step20.show(truncate=False)

+-------+--------------------------+--------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|source |rejection_reason          |rejection_timestamp       |original_data                                                                                                                                                                      |
+-------+--------------------------+--------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|reviews|Note hors intervalle [1,5]|2026-07-24 08:00:28.868557|{"_id":"REV001","comment":"Très bon produit","customerId":"cust-26","orderId":" ORD33","productId":"P009","rating":7,"reviewDate":"2026-03-12T10:25:00","verifiedPurchase":"yes"}  |
|reviews|Note hors inter

In [288]:
reviews_valid = (
    reviews_step20
    .filter(col("is_rating_valid") == True))

In [289]:
reviews_valid.select(
    "rating",
    "is_rating_valid",
    "comment"
).show(30, truncate=False)

+------+---------------+----------------+
|rating|is_rating_valid|comment         |
+------+---------------+----------------+
|3     |true           |Très bon produit|
|5     |true           |Très bon produit|
|5     |true           |Très bon produit|
|2     |true           |Très bon produit|
|2     |true           |Très bon produit|
|2     |true           |Très bon produit|
|5     |true           |NULL            |
|5     |true           |Très bon produit|
|1     |true           |Très bon produit|
|2     |true           |Très bon produit|
|4     |true           |Très bon produit|
|5     |true           |NULL            |
|5     |true           |Très bon produit|
|1     |true           |Très bon produit|
|5     |true           |NULL            |
|3     |true           |Très bon produit|
|4     |true           |Très bon produit|
|2     |true           |Très bon produit|
|5     |true           |Très bon produit|
|4     |true           |NULL            |
+------+---------------+----------

#### Étape 21 : déduplication des avis

In [290]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, length

In [291]:
reviews_step21_tmp = (
    reviews_valid
    .withColumn(
        "comment_length",
        when(
            col("comment").isNull(),
            0)
    .otherwise(
            length(col("comment")))))

In [292]:
window_reviews = (
    Window.partitionBy(
        "customer_id_clean",
        "product_id_clean",
        "order_id_clean")
    .orderBy(
        col("review_date").desc(),
        col("comment_length").desc()))

In [293]:
reviews_step21_tmp = (reviews_step21_tmp.withColumn("row_num",row_number().over(window_reviews)))

In [294]:
reviews_step21_tmp.select(
    "customer_id_clean",
    "product_id_clean",
    "order_id_clean",
    "review_date",
    "comment_length",
    "row_num"
).show(30, truncate=False)

+-----------------+----------------+--------------+-------------------+--------------+-------+
|customer_id_clean|product_id_clean|order_id_clean|review_date        |comment_length|row_num|
+-----------------+----------------+--------------+-------------------+--------------+-------+
|C000001          |P011            |ORD000028     |2026-03-12 10:25:00|16            |1      |
|C000002          |P003            |ORD000009     |2026-03-12 10:25:00|16            |1      |
|C000003          |P013            |ORD000019     |2026-03-12 10:25:00|16            |1      |
|C000006          |P011            |ORD000018     |2026-03-12 10:25:00|16            |1      |
|C000010          |P014            |ORD000013     |2026-03-12 10:25:00|16            |1      |
|C000014          |P012            |ORD000040     |2026-03-12 10:25:00|0             |1      |
|C000014          |P015            |ORD000028     |2026-03-12 10:25:00|16            |1      |
|C000016          |P005            |ORD000026     

In [295]:
reviews_clean = (
    reviews_step21_tmp
    .filter(col("row_num") == 1)
    .drop(
        "row_num",
        "comment_length"))

In [296]:
reviews_clean.groupBy(
    "customer_id_clean",
    "product_id_clean",
    "order_id_clean"
).count().filter(
    col("count") > 1
).show()

+-----------------+----------------+--------------+-----+
|customer_id_clean|product_id_clean|order_id_clean|count|
+-----------------+----------------+--------------+-----+
+-----------------+----------------+--------------+-----+



#### Étape 22 : contrôle des achats vérifiés

In [297]:
reviews_step22 = (
    reviews_clean.alias("r")
    .join(
        orders_clean.select(
            "order_id_clean",
            "customer_id_clean"
        ).alias("o"),
        on="order_id_clean",
        how="left"))

In [298]:
reviews_step22 = (
    reviews_step22
    .withColumn(
        "order_exists",
        col("o.order_id_clean").isNotNull())
    .withColumn(
        "customer_order_match",
        col("r.customer_id_clean") == col("o.customer_id_clean")))

In [299]:
reviews_step22 = (
    reviews_step22
    .withColumn(
        "verified_purchase_computed",
        col("order_exists")
        &
        col("customer_order_match")))

In [300]:
reviews_step22 = (
    reviews_step22
    .withColumn(
        "is_verification_consistent",
        col("verified_purchase")
        == col("verified_purchase_computed")))

In [301]:
reviews_step22.select(
    col("r.customer_id_clean").alias("customer_id_clean"),
    col("order_id_clean"),
    col("verified_purchase"),
    col("verified_purchase_computed"),
    col("is_verification_consistent")
).show(20, truncate=False)

+-----------------+--------------+-----------------+--------------------------+--------------------------+
|customer_id_clean|order_id_clean|verified_purchase|verified_purchase_computed|is_verification_consistent|
+-----------------+--------------+-----------------+--------------------------+--------------------------+
|C000018          |ORD000027     |false            |false                     |true                      |
|C000017          |ORD000017     |false            |false                     |true                      |
|C000019          |ORD000017     |false            |false                     |true                      |
|C000001          |ORD000028     |false            |false                     |true                      |
|C000014          |ORD000028     |true             |false                     |false                     |
|C000002          |ORD000009     |false            |false                     |true                      |
|C000020          |ORD000009     |tru

## Partie 9 — Nettoyage des événements JSON

#### Étape 23 : aplatissement du JSON

In [302]:
delivery_step23 = (
    delivery_events_df
    .select(
        "event_id",
        "order_id",
        "event_type",
        "event_timestamp",

        col("location.city").alias("delivery_city"),
        col("location.country").alias("delivery_country"),

        col("carrier.id").alias("carrier_id"),
        col("carrier.name").alias("carrier_name")))

In [303]:
delivery_step23.printSchema()
delivery_step23.show(20, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- delivery_city: string (nullable = true)
 |-- delivery_country: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- carrier_name: string (nullable = true)

+--------+--------+-------------+-------------------+-------------+----------------+----------+------------+
|event_id|order_id|event_type   |event_timestamp    |delivery_city|delivery_country|carrier_id|carrier_name|
+--------+--------+-------------+-------------------+-------------+----------------+----------+------------+
|EVT001  |ord-0036|X            |2026-03-17T08:30:00|PARIS        |France          |DHL01     |UPS         |
|EVT002  |ord-0029|X            |2026-03-04T08:30:00|Paris        |France          |UPS01     |DHL         |
|EVT003  | ORD25  |X            |2026-03-12T08:30:00|Paris        |France          |DHL01     |UPS   

#### Étape 24 : normalisation des événements

In [304]:
delivery_step24 = (
    delivery_step23
    .withColumn(
        "order_id_tmp",
        upper(trim(col("order_id"))))
    .withColumn(
        "order_id_tmp",
        regexp_replace(
            col("order_id_tmp"),
            "[-_\\s]",
            ""))
    .withColumn(
        "order_id_number",
        regexp_extract(
            col("order_id_tmp"),
            "^ORD0*([0-9]+)$",
            1))
    .withColumn(
        "order_id_clean",
        when(
            col("order_id_number") != "",
            concat(
                lit("ORD"),
                lpad(col("order_id_number"), 6, "0"))))
    .withColumn(
        "event_type_clean",
        upper(trim(col("event_type"))))
    .withColumn(
        "event_timestamp_clean",
        to_timestamp(col("event_timestamp")))
    .withColumn(
        "carrier_name_clean",
        initcap(trim(col("carrier_name"))))
    .withColumn(
        "delivery_city",
        initcap(trim(col("delivery_city"))))
    .withColumn(
        "delivery_country",
        initcap(trim(col("delivery_country")))))

In [305]:
delivery_step24 = (
    delivery_step24
    .withColumn(
        "event_type_clean",
        when(
            col("event_type_clean").isin(
                "ORDER_CREATED",
                "PREPARING",
                "SHIPPED",
                "IN_TRANSIT",
                "DELIVERED",
                "RETURNED"
            ),
            col("event_type_clean")
        )
        .otherwise("UNKNOWN")))

In [306]:
delivery_step24.select(
    "order_id",
    "order_id_clean",
    "event_type",
    "event_type_clean",
    "event_timestamp",
    "event_timestamp_clean"
).show(20, truncate=False)

+--------+--------------+-------------+----------------+-------------------+---------------------+
|order_id|order_id_clean|event_type   |event_type_clean|event_timestamp    |event_timestamp_clean|
+--------+--------------+-------------+----------------+-------------------+---------------------+
|ord-0036|ORD000036     |X            |UNKNOWN         |2026-03-17T08:30:00|2026-03-17 08:30:00  |
|ord-0029|ORD000029     |X            |UNKNOWN         |2026-03-04T08:30:00|2026-03-04 08:30:00  |
| ORD25  |ORD000025     |X            |UNKNOWN         |2026-03-12T08:30:00|2026-03-12 08:30:00  |
|ORD-0018|ORD000018     |ORDER_CREATED|ORDER_CREATED   |2026-03-26T08:30:00|2026-03-26 08:30:00  |
|ord-0014|ORD000014     |X            |UNKNOWN         |2026-03-11T08:30:00|2026-03-11 08:30:00  |
|ord-0039|ORD000039     |RETURNED     |RETURNED        |2026-03-16T08:30:00|2026-03-16 08:30:00  |
|ord-0011|ORD000011     |RETURNED     |RETURNED        |2026-03-24T08:30:00|2026-03-24 08:30:00  |
|ord-0022|

#### Étape 25 : suppression des événements dupliqués

In [307]:
delivery_step25 = (
    delivery_step24
    .dropDuplicates(
        [
            "order_id_clean",
            "event_type_clean",
            "event_timestamp_clean",
            "carrier_id"
        ]))

In [308]:
print("Avant :", delivery_step24.count())
print("Après :", delivery_step25.count())

Avant : 60
Après : 59


In [309]:
delivery_step25.select(
    "order_id_clean",
    "event_type_clean",
    "event_timestamp_clean",
    "carrier_id"
).show(20, truncate=False)

+--------------+----------------+---------------------+----------+
|order_id_clean|event_type_clean|event_timestamp_clean|carrier_id|
+--------------+----------------+---------------------+----------+
|ORD000036     |ORDER_CREATED   |2026-03-20 08:30:00  |DHL01     |
|ORD000001     |PREPARING       |2026-03-03 08:30:00  |UPS01     |
|ORD000031     |PREPARING       |2026-03-28 08:30:00  |DHL01     |
|ORD000038     |PREPARING       |2026-03-04 08:30:00  |UPS01     |
|ORD000033     |RETURNED        |2026-03-10 08:30:00  |DHL01     |
|ORD000011     |DELIVERED       |2026-03-19 08:30:00  |UPS01     |
|ORD000008     |SHIPPED         |2026-03-23 08:30:00  |DHL01     |
|ORD000014     |UNKNOWN         |2026-03-11 08:30:00  |UPS01     |
|ORD000033     |UNKNOWN         |2026-03-03 08:30:00  |UPS01     |
|ORD000029     |UNKNOWN         |2026-03-04 08:30:00  |UPS01     |
|ORD000026     |PREPARING       |2026-03-03 08:30:00  |DHL01     |
|ORD000010     |PREPARING       |2026-03-27 08:30:00  |UPS01  

In [310]:
delivery_step25.groupBy(
    "order_id_clean",
    "event_type_clean",
    "event_timestamp_clean",
    "carrier_id"
).count().filter(
    col("count") > 1
).show()

+--------------+----------------+---------------------+----------+-----+
|order_id_clean|event_type_clean|event_timestamp_clean|carrier_id|count|
+--------------+----------------+---------------------+----------+-----+
+--------------+----------------+---------------------+----------+-----+



#### Étape 26 : détermination du dernier statut

In [311]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number,
    first,
    last,
    min,
    max,
    count)

In [312]:
window_delivery = (
    Window
    .partitionBy("order_id_clean")
    .orderBy(col("event_timestamp_clean").desc()))

In [313]:
delivery_ranked = (
    delivery_step25
    .withColumn(
        "row_num",
        row_number().over(window_delivery)))

In [314]:
last_delivery_event = (
    delivery_ranked
    .filter(col("row_num") == 1))

In [315]:
delivery_summary = (
    delivery_step25
    .groupBy("order_id_clean")
    .agg(
        min("event_timestamp_clean").alias("first_event"),
        max("event_timestamp_clean").alias("last_event"),
        count("*").alias("number_of_events")))

In [316]:
delivery_summary = (
    delivery_summary
    .join(
        last_delivery_event.select(
            "order_id_clean",
            "event_type_clean",
            "carrier_name_clean",
            "delivery_city"),
        on="order_id_clean",
        how="left"))

In [317]:
delivery_summary = (
    delivery_summary
    .withColumnRenamed(
        "event_type_clean",
        "last_delivery_status"))

In [318]:
delivery_summary.show(20, truncate=False)

+--------------+-------------------+-------------------+----------------+--------------------+------------------+-------------+
|order_id_clean|first_event        |last_event         |number_of_events|last_delivery_status|carrier_name_clean|delivery_city|
+--------------+-------------------+-------------------+----------------+--------------------+------------------+-------------+
|ORD000002     |2026-03-26 08:30:00|2026-03-26 08:30:00|1               |DELIVERED           |Ups               |Toulouse     |
|ORD000027     |2026-03-02 08:30:00|2026-03-11 08:30:00|2               |IN_TRANSIT          |Ups               |Paris        |
|ORD000023     |2026-03-09 08:30:00|2026-03-19 08:30:00|3               |RETURNED            |Ups               |Toulouse     |
|ORD000036     |2026-03-17 08:30:00|2026-03-20 08:30:00|3               |ORDER_CREATED       |Ups               |Toulouse     |
|ORD000015     |2026-03-02 08:30:00|2026-03-02 08:30:00|1               |SHIPPED             |Ups       

#### Étape 27 : calcul du délai de livraison

In [319]:
delivery_summary = (
    delivery_summary
    .join(
        orders_clean.select(
            "order_id_clean",
            "order_date_clean"),
        on="order_id_clean",
        how="left"))

In [320]:
from pyspark.sql.functions import datediff

In [321]:
delivery_summary = (
    delivery_summary
    .withColumn(
        "delivery_delay_days",
        datediff(
            col("last_event"),
            col("order_date_clean"))))

In [322]:
delivery_summary = (
    delivery_summary
    .withColumn(
        "delivery_performance",
        when(
            col("delivery_delay_days").between(0, 2),
            "Très rapide")
        .when(
            col("delivery_delay_days").between(3, 5),
            "Normal")
        .when(
            col("delivery_delay_days").between(6, 10),
            "Lent")
        .when(
            col("delivery_delay_days") > 10,
            "Très lent")
        .otherwise("Non livré")))

In [323]:
delivery_summary.select(
    "order_id_clean",
    "last_delivery_status",
    "delivery_delay_days",
    "delivery_performance"
).show(30, truncate=False)

+--------------+--------------------+-------------------+--------------------+
|order_id_clean|last_delivery_status|delivery_delay_days|delivery_performance|
+--------------+--------------------+-------------------+--------------------+
|ORD000002     |DELIVERED           |NULL               |Non livré           |
|ORD000027     |IN_TRANSIT          |NULL               |Non livré           |
|ORD000023     |RETURNED            |NULL               |Non livré           |
|ORD000036     |ORDER_CREATED       |19                 |Très lent           |
|ORD000015     |SHIPPED             |1                  |Très rapide         |
|ORD000017     |RETURNED            |NULL               |Non livré           |
|ORD000028     |DELIVERED           |8                  |Lent                |
|ORD000018     |ORDER_CREATED       |NULL               |Non livré           |
|ORD000004     |DELIVERED           |2                  |Très rapide         |
|ORD000003     |PREPARING           |NULL           

## Partie 10 — Croisement des différentes sources


#### Étape 28 : jointure entre les clients et les commandes

In [324]:
jointure_clients_commandes = (
    orders_clean.alias("o")
    .join(
        customers_clean.alias("c"),
        on="customer_id_clean",
        how="left")
    .withColumn(
        "customer_found",
        col("c.customer_id_clean").isNotNull()))

In [325]:
jointure_clients_commandes.select(
    "order_id_clean",
    "customer_id_clean",
    "full_name",
    "customer_found"
).show(20, truncate=False)

+--------------+-----------------+--------------+--------------+
|order_id_clean|customer_id_clean|full_name     |customer_found|
+--------------+-----------------+--------------+--------------+
|ORD000014     |C000009          |Prenom9 Nom9  |true          |
|ORD000007     |C000002          |Prenom2 Nom2  |true          |
|ORD000038     |C000011          |Prenom11 Nom11|true          |
|ORD000015     |C000027          |Prenom27 Nom27|true          |
|ORD000020     |C000016          |Prenom16 Nom16|true          |
|ORD000001     |C000999          |NULL          |false         |
|ORD000024     |C000019          |Prenom19 Nom19|true          |
|ORD000004     |C000004          |Prenom4 Nom4  |true          |
|ORD000028     |C000015          |Prenom15 Nom15|true          |
|ORD000029     |C000010          |Prenom10 Nom10|true          |
|ORD000036     |C000010          |Prenom10 Nom10|true          |
|ORD000013     |C000022          |Prenom22 Nom22|true          |
|ORD000011     |C000007  

In [326]:
commandes_orphelines = (
    jointure_clients_commandes
    .filter(col("customer_found") == False))

print(
    "Nombre de commandes orphelines :",
    commandes_orphelines.count())


Nombre de commandes orphelines : 1


#### Étape 29 : jointure avec les lignes de commande

In [327]:
jointure_commandes_articles = (
    jointure_clients_commandes
    .join(
        order_items_summary,
        on="order_id_clean",
        how="left"))

In [328]:
jointure_commandes_articles = (
    jointure_commandes_articles
    .withColumn(
        "amount_difference",
        abs(
            col("total_amount_eur") - col("net_amount_total"))))

In [329]:
jointure_commandes_articles.select(
    "order_id_clean",
    "number_of_products",
    "total_quantity",
    "gross_amount_total",
    "discount_total",
    "net_amount_total",
    "amount_difference"
).show(20, truncate=False)

+--------------+------------------+--------------+------------------+--------------+----------------+-----------------+
|order_id_clean|number_of_products|total_quantity|gross_amount_total|discount_total|net_amount_total|amount_difference|
+--------------+------------------+--------------+------------------+--------------+----------------+-----------------+
|ORD000001     |NULL              |NULL          |NULL              |NULL          |NULL            |NULL             |
|ORD000004     |1                 |2             |50.0000000        |5.000000      |45.000000       |55.5             |
|ORD000007     |1                 |5             |350.0000000       |65.000000     |285.000000      |233.57           |
|ORD000011     |NULL              |NULL          |NULL              |NULL          |NULL            |NULL             |
|ORD000013     |2                 |3             |75.0000000        |12.500000     |62.500000       |53.25            |
|ORD000014     |NULL              |NULL 

#### Étape 30 : jointure avec les produits

In [331]:
from pyspark.sql.functions import (
    collect_set,
    collect_list,
    countDistinct)

In [332]:
order_items_step16 = (
    order_items_step16
    .withColumn(
        "product_id_clean",
        upper(
            regexp_replace(
                trim(col("product_id")),
                "[-_\\s]",""))))

In [333]:
details_produits_commande = (
    order_items_step16.alias("oi")
    .join(
        products_clean.alias("p"),
        on="product_id_clean",
        how="left")
    .groupBy("order_id_clean")
    .agg(
        collect_set("product_name_clean").alias("liste_produits"),
        collect_set("category_clean").alias("liste_categories"),
        collect_set("brand_clean").alias("liste_marques"),
        countDistinct("category_clean").alias(
            "nombre_categories_differentes")))

In [334]:
details_produits_commande.show(
    20,truncate=False)

+--------------+----------------------------------+----------------+------------------+-----------------------------+
|order_id_clean|liste_produits                    |liste_categories|liste_marques     |nombre_categories_differentes|
+--------------+----------------------------------+----------------+------------------+-----------------------------+
|ORD000027     |[Produit 4]                       |[Technologie]   |[Lenovo]          |1                            |
|ORD000036     |[Produit 9]                       |[Technologie]   |[Lenovo]          |1                            |
|ORD000017     |[Produit 10]                      |[Technologie]   |[Dell]            |1                            |
|ORD000009     |[Produit 8]                       |[Technologie]   |[Lenovo]          |1                            |
|ORD000032     |[Produit 6]                       |[Technologie]   |[Hp]              |1                            |
|ORD000004     |[Produit 9]                       |[Tech

In [335]:
jointure_commandes_produits = (
    jointure_commandes_articles
    .join(
        details_produits_commande,
        on="order_id_clean",
        how="left"))

In [336]:
jointure_commandes_produits.select(
    "order_id_clean",
    "liste_produits",
    "liste_categories",
    "liste_marques",
    "nombre_categories_differentes"
).show(20, truncate=False)

+--------------+----------------------+----------------+-------------+-----------------------------+
|order_id_clean|liste_produits        |liste_categories|liste_marques|nombre_categories_differentes|
+--------------+----------------------+----------------+-------------+-----------------------------+
|ORD000001     |NULL                  |NULL            |NULL         |NULL                         |
|ORD000004     |[Produit 9]           |[Technologie]   |[Lenovo]     |1                            |
|ORD000007     |[Produit 13]          |[Technologie]   |[Lenovo]     |1                            |
|ORD000011     |NULL                  |NULL            |NULL         |NULL                         |
|ORD000013     |[Produit 2, Produit 5]|[Technologie]   |[Dell, Hp]   |1                            |
|ORD000014     |NULL                  |NULL            |NULL         |NULL                         |
|ORD000015     |NULL                  |NULL            |NULL         |NULL                 

#### Étape 31 : jointure avec les avis MongoDB

In [337]:
from pyspark.sql.functions import (
    avg,
    min,
    max,
    count,
    sum as spark_sum)

In [338]:
resume_avis_commandes = (
    reviews_clean
    .groupBy("order_id_clean")
    .agg(
        count("*").alias("nombre_avis"),
        avg("rating").alias("note_moyenne"),
        min("rating").alias("note_min"),
        max("rating").alias("note_max"),
        spark_sum(
            when(
                col("verified_purchase") == True,
                1).otherwise(0)).alias("nombre_achats_verifies"),
        spark_sum(
            when(
                col("comment").isNotNull(),
                1
            ).otherwise(0)
        ).alias("nombre_commentaires"),
        (spark_sum(
                when(
                    col("rating") >= 4,
                    1).otherwise(0))/count("*")* 100).alias("pourcentage_avis_positifs")))

In [339]:
resume_avis_commandes = (
    resume_avis_commandes
    .withColumn(
        "customer_satisfaction",
        when(
            col("note_moyenne") >= 4,
            "Très satisfait")
        .when(
            col("note_moyenne") >= 3,
            "Satisfait")
        .when(
            col("note_moyenne") >= 2,
            "Peu satisfait")
        .when(
            col("note_moyenne") < 2,
            "Insatisfait")
        .otherwise("Non évalué")))

In [340]:
resume_avis_commandes.show(
    20,
    truncate=False)

+--------------+-----------+------------------+--------+--------+----------------------+-------------------+-------------------------+---------------------+
|order_id_clean|nombre_avis|note_moyenne      |note_min|note_max|nombre_achats_verifies|nombre_commentaires|pourcentage_avis_positifs|customer_satisfaction|
+--------------+-----------+------------------+--------+--------+----------------------+-------------------+-------------------------+---------------------+
|ORD000027     |1          |5.0               |5       |5       |0                     |1                  |100.0                    |Très satisfait       |
|ORD000017     |2          |3.5               |2       |5       |0                     |2                  |50.0                     |Satisfait            |
|ORD000028     |2          |4.5               |4       |5       |1                     |2                  |100.0                    |Très satisfait       |
|ORD000009     |3          |2.6666666666666665|1       |5 

In [341]:
jointure_commandes_avis = (
    jointure_commandes_produits
    .join(
        resume_avis_commandes,
        on="order_id_clean",
        how="left"))

In [342]:
jointure_commandes_avis.select(
    "order_id_clean",
    "nombre_avis",
    "note_moyenne",
    "note_min",
    "note_max",
    "nombre_achats_verifies",
    "nombre_commentaires",
    "pourcentage_avis_positifs",
    "customer_satisfaction"
).show(20, truncate=False)

+--------------+-----------+------------+--------+--------+----------------------+-------------------+-------------------------+---------------------+
|order_id_clean|nombre_avis|note_moyenne|note_min|note_max|nombre_achats_verifies|nombre_commentaires|pourcentage_avis_positifs|customer_satisfaction|
+--------------+-----------+------------+--------+--------+----------------------+-------------------+-------------------------+---------------------+
|ORD000001     |NULL       |NULL        |NULL    |NULL    |NULL                  |NULL               |NULL                     |NULL                 |
|ORD000004     |NULL       |NULL        |NULL    |NULL    |NULL                  |NULL               |NULL                     |NULL                 |
|ORD000007     |1          |2.0         |2       |2       |0                     |1                  |0.0                      |Peu satisfait        |
|ORD000011     |1          |5.0         |5       |5       |0                     |0           

#### Étape 32 : jointure avec les événements de livraison

In [343]:
jointure_commandes_livraisons = (
    jointure_commandes_avis
    .join(
        delivery_summary,
        on="order_id_clean",
        how="left")
    .withColumn(
        "delivery_found",
        col("last_delivery_status").isNotNull()))

In [344]:
jointure_commandes_livraisons.select(
    "order_id_clean",
    "last_delivery_status",
    "carrier_name_clean",
    "first_event",
    "last_event",
    "delivery_delay_days",
    "delivery_performance",
    "delivery_found"
).show(20, False)

+--------------+--------------------+------------------+-------------------+-------------------+-------------------+--------------------+--------------+
|order_id_clean|last_delivery_status|carrier_name_clean|first_event        |last_event         |delivery_delay_days|delivery_performance|delivery_found|
+--------------+--------------------+------------------+-------------------+-------------------+-------------------+--------------------+--------------+
|ORD000001     |UNKNOWN             |Ups               |2026-03-03 08:30:00|2026-03-16 08:30:00|15                 |Très lent           |true          |
|ORD000004     |DELIVERED           |Dhl               |2026-03-03 08:30:00|2026-03-03 08:30:00|2                  |Très rapide         |true          |
|ORD000007     |NULL                |NULL              |NULL               |NULL               |NULL               |NULL                |false         |
|ORD000011     |RETURNED            |Dhl               |2026-03-05 08:30:00|2026-0

## Partie 11 — Construction de la vue consolidée

#### Étape 33 : création de customer_order_360

In [345]:
customer_order_360 = (
    jointure_commandes_livraisons.select(
        "order_id_clean",
        "customer_id_clean",
        "full_name",
        "email_clean",
        "phone_clean",
        "city_clean",
        "country_clean",

        col("o.order_date_clean").alias("order_date_clean"),

        "order_status",
        "payment_method",
        "currency",
        "total_amount_eur",
        "number_of_products",
        "total_quantity",
        "gross_amount_total",
        "discount_total",
        "net_amount_total",
        "amount_difference",
        "liste_produits",
        "liste_categories",
        "liste_marques",
        "nombre_categories_differentes",
        "nombre_avis",
        "note_moyenne",
        "note_min",
        "note_max",
        "pourcentage_avis_positifs",
        "customer_satisfaction",
        "last_delivery_status",
        "carrier_name_clean",
        "first_event",
        "last_event",
        "delivery_delay_days",
        "delivery_performance"))

In [346]:
customer_order_360.printSchema()

root
 |-- order_id_clean: string (nullable = true)
 |-- customer_id_clean: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email_clean: string (nullable = true)
 |-- phone_clean: string (nullable = true)
 |-- city_clean: string (nullable = true)
 |-- country_clean: string (nullable = true)
 |-- order_date_clean: timestamp (nullable = true)
 |-- order_status: string (nullable = false)
 |-- payment_method: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- total_amount_eur: double (nullable = true)
 |-- number_of_products: long (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- gross_amount_total: decimal(38,7) (nullable = true)
 |-- discount_total: decimal(38,6) (nullable = true)
 |-- net_amount_total: decimal(38,6) (nullable = true)
 |-- amount_difference: double (nullable = true)
 |-- liste_produits: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- liste_categories: array (nullable = true)
 |    |--

#### Étape 34 : calcul d’un score de qualité

In [347]:
customer_order_360 = (
    customer_order_360
    .withColumn(
        "data_quality_score",
        lit(100)
        - when(col("amount_difference") > 0.01, 20).otherwise(0)
        - when(col("delivery_delay_days").isNull(), 10).otherwise(0)
        - when(col("email_clean").isNull(), 10).otherwise(0))
    .withColumn(
        "quality_level",
        when(col("data_quality_score") >= 90, "Excellent")
        .when(col("data_quality_score") >= 75, "Bon")
        .when(col("data_quality_score") >= 50, "Moyen")
        .otherwise("Faible")))

In [348]:
customer_order_360.select(
    "order_id_clean",
    "data_quality_score",
    "quality_level"
).show(20, False)

+--------------+------------------+-------------+
|order_id_clean|data_quality_score|quality_level|
+--------------+------------------+-------------+
|ORD000001     |90                |Excellent    |
|ORD000004     |80                |Bon          |
|ORD000007     |70                |Moyen        |
|ORD000011     |90                |Excellent    |
|ORD000013     |80                |Bon          |
|ORD000014     |100               |Excellent    |
|ORD000015     |100               |Excellent    |
|ORD000020     |70                |Moyen        |
|ORD000024     |90                |Excellent    |
|ORD000026     |90                |Excellent    |
|ORD000028     |100               |Excellent    |
|ORD000029     |100               |Excellent    |
|ORD000036     |80                |Bon          |
|ORD000038     |100               |Excellent    |
+--------------+------------------+-------------+



## Partie 12 — Contrôles de cohérence


#### Étape 35 : vérification du résultat final

In [349]:
#Doublons
customer_order_360.groupBy(
    "order_id_clean"
).count().filter(
    col("count") > 1
).show()

+--------------+-----+
|order_id_clean|count|
+--------------+-----+
+--------------+-----+



In [350]:
#Montants négatifs
customer_order_360.filter(
    col("total_amount_eur") < 0
).show()


+--------------+-----------------+---------+-----------+-----------+----------+-------------+----------------+------------+--------------+--------+----------------+------------------+--------------+------------------+--------------+----------------+-----------------+--------------+----------------+-------------+-----------------------------+-----------+------------+--------+--------+-------------------------+---------------------+--------------------+------------------+-----------+----------+-------------------+--------------------+------------------+-------------+
|order_id_clean|customer_id_clean|full_name|email_clean|phone_clean|city_clean|country_clean|order_date_clean|order_status|payment_method|currency|total_amount_eur|number_of_products|total_quantity|gross_amount_total|discount_total|net_amount_total|amount_difference|liste_produits|liste_categories|liste_marques|nombre_categories_differentes|nombre_avis|note_moyenne|note_min|note_max|pourcentage_avis_positifs|customer_satisfa

In [351]:
#Score hors bornes 
customer_order_360.filter(
    (col("data_quality_score") < 0)
    | (col("data_quality_score") > 100)
).show()

+--------------+-----------------+---------+-----------+-----------+----------+-------------+----------------+------------+--------------+--------+----------------+------------------+--------------+------------------+--------------+----------------+-----------------+--------------+----------------+-------------+-----------------------------+-----------+------------+--------+--------+-------------------------+---------------------+--------------------+------------------+-----------+----------+-------------------+--------------------+------------------+-------------+
|order_id_clean|customer_id_clean|full_name|email_clean|phone_clean|city_clean|country_clean|order_date_clean|order_status|payment_method|currency|total_amount_eur|number_of_products|total_quantity|gross_amount_total|discount_total|net_amount_total|amount_difference|liste_produits|liste_categories|liste_marques|nombre_categories_differentes|nombre_avis|note_moyenne|note_min|note_max|pourcentage_avis_positifs|customer_satisfa

In [353]:
#Volume final

print("Nombre de lignes customer_order_360 :",customer_order_360.count())

Nombre de lignes customer_order_360 : 14


In [354]:
#Nombre d'anomalies par contrôle

nb_doublons = (
    customer_order_360
    .groupBy("order_id_clean")
    .count()
    .filter(col("count") > 1)
    .count())

nb_montants_negatifs = (
    customer_order_360
    .filter(col("total_amount_eur") < 0)
    .count())

nb_notes_invalides = (
    customer_order_360
    .filter(
        col("note_moyenne").isNotNull() &
        ((col("note_moyenne") < 1) | (col("note_moyenne") > 5))).count())

nb_scores_invalides = (
    customer_order_360
    .filter((col("data_quality_score") < 0)|(col("data_quality_score") > 100)).count())

nb_dates_invalides = (
    customer_order_360
    .filter(col("last_event").isNotNull() & (col("last_event") < col("order_date_clean"))).count())

nb_annulees_livrees = (
    customer_order_360
    .filter((col("order_status") == "CANCELLED") & (col("last_delivery_status") == "DELIVERED")).count())

nb_livrees_sans_date = (
    customer_order_360
    .filter((col("last_delivery_status") == "DELIVERED") & col("last_event").isNull()).count())

nb_statuts_invalides = (
    customer_order_360
    .filter(
        ~col("order_status").isin(
            "CREATED",
            "PAID",
            "PREPARING",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED",
            "RETURNED",
            "UNKNOWN")).count())

In [355]:
validation_results = spark.createDataFrame(
    [
        ("Unicité des commandes", nb_doublons),
        ("Montants négatifs", nb_montants_negatifs),
        ("Notes hors intervalle [1,5]", nb_notes_invalides),
        ("Scores qualité invalides", nb_scores_invalides),
        ("Livraison avant commande", nb_dates_invalides),
        ("Commandes annulées livrées", nb_annulees_livrees),
        ("Livrées sans date de livraison", nb_livrees_sans_date),
        ("Statuts non normalisés", nb_statuts_invalides)
    ],
    ["controle", "nombre_anomalies"])

In [356]:
validation_results = (
    validation_results
    .withColumn(
        "statut",
        when(col("nombre_anomalies") == 0, "OK").otherwise("ERREUR")))

In [357]:
validation_results.show(truncate=False)

+------------------------------+----------------+------+
|controle                      |nombre_anomalies|statut|
+------------------------------+----------------+------+
|Unicité des commandes         |0               |OK    |
|Montants négatifs             |0               |OK    |
|Notes hors intervalle [1,5]   |0               |OK    |
|Scores qualité invalides      |0               |OK    |
|Livraison avant commande      |0               |OK    |
|Commandes annulées livrées    |2               |ERREUR|
|Livrées sans date de livraison|0               |OK    |
|Statuts non normalisés        |0               |OK    |
+------------------------------+----------------+------+



## Partie 13 — Gestion des rejets


#### Étape 36 : création d’une zone de rejet

In [358]:
orders_rejects_step15 = (
    orders_step15
    .filter(
        (col("total_amount") <= 0)
        |
        col("currency").isNull()
    )
    .select(
        lit("orders").alias("source"),
        lit("Montant ou devise invalide").alias("rejection_reason"),
        current_timestamp().alias("rejection_timestamp"),
        to_json(
            struct([col(c) for c in orders_step15.columns])
        ).alias("original_data")))

In [359]:
table_rejets = (
    orders_rejects_step13
    .unionByName(orders_rejects_step15)
    .unionByName(order_items_rejects_step16)
    .unionByName(reviews_rejects_step20))

In [360]:
print("Nombre total de rejets :", table_rejets.count())

Nombre total de rejets : 92


In [363]:
orders_rejects_step13.write.mode("overwrite").parquet(
    "output/rejects/orders_dates")

orders_rejects_step15.write.mode("overwrite").parquet(
    "output/rejects/orders")

order_items_rejects_step16.write.mode("overwrite").parquet(
    "output/rejects/order_items")

reviews_rejects_step20.write.mode("overwrite").parquet(
    "output/rejects/reviews")

In [364]:
table_rejets.write.mode("overwrite").parquet(
    "output/rejects/all_rejects")

## Partie 14 — Enregistrement dans une source unique


#### Étape 37 : écriture du résultat final

In [365]:
customer_order_360.write.mode("overwrite").parquet(
    "output/customer_order_360")

validation_results.write.mode("overwrite").parquet(
    "output/validation_results")

table_rejets.write.mode("overwrite").parquet(
    "output/table_rejets")

In [366]:
from pyspark.sql.functions import year, month

In [367]:
customer_order_360_partitionne = (
    customer_order_360
    .withColumn(
        "order_year",
        year(col("order_date_clean")))
    .withColumn(
        "order_month",
        month(col("order_date_clean"))))

In [368]:
customer_order_360_partitionne.select(
    "order_date_clean",
    "order_year",
    "order_month"
).show(10, False)

+-------------------+----------+-----------+
|order_date_clean   |order_year|order_month|
+-------------------+----------+-----------+
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
|2026-03-01 00:00:00|2026      |3          |
+-------------------+----------+-----------+
only showing top 10 rows



In [369]:
customer_order_360_partitionne = (
    customer_order_360_partitionne
    .coalesce(1))

In [370]:
customer_order_360_partitionne.write \
    .mode("overwrite") \
    .partitionBy(
        "order_year",
        "order_month") \
    .parquet(
        "output/customer_order_360")

In [371]:
customer_order_360_parquet = spark.read.parquet(
    "output/customer_order_360")

print("Nombre de lignes :",customer_order_360_parquet.count())

Nombre de lignes : 14


In [372]:
customer_order_360_parquet.show(5,truncate=False)

+--------------+-----------------+--------------+------------------+------------+----------+-------------+-------------------+------------+--------------+--------+----------------+------------------+--------------+------------------+--------------+----------------+-----------------+----------------------+----------------+-------------+-----------------------------+-----------+------------+--------+--------+-------------------------+---------------------+--------------------+------------------+-------------------+-------------------+-------------------+--------------------+------------------+-------------+----------+-----------+
|order_id_clean|customer_id_clean|full_name     |email_clean       |phone_clean |city_clean|country_clean|order_date_clean   |order_status|payment_method|currency|total_amount_eur|number_of_products|total_quantity|gross_amount_total|discount_total|net_amount_total|amount_difference|liste_produits        |liste_categories|liste_marques|nombre_categories_different

#### Étape 38 : rechargement du résultat

In [373]:
# Lecture des fichiers Parquet

customer_order_360_parquet = spark.read.parquet(
    "output/customer_order_360")

validation_results_parquet = spark.read.parquet(
    "output/validation_results")

table_rejets_parquet = spark.read.parquet(
    "output/table_rejets")

In [374]:
#Vérification des volumes

print("Customer Order 360 :",customer_order_360_parquet.count())

print("Validation Results :",validation_results_parquet.count())

print("Table Rejets :",table_rejets_parquet.count())

Customer Order 360 : 14
Validation Results : 8
Table Rejets : 92


In [375]:
# Vérification des schémas

customer_order_360_parquet.printSchema()

root
 |-- order_id_clean: string (nullable = true)
 |-- customer_id_clean: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email_clean: string (nullable = true)
 |-- phone_clean: string (nullable = true)
 |-- city_clean: string (nullable = true)
 |-- country_clean: string (nullable = true)
 |-- order_date_clean: timestamp (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- total_amount_eur: double (nullable = true)
 |-- number_of_products: long (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- gross_amount_total: decimal(38,7) (nullable = true)
 |-- discount_total: decimal(38,6) (nullable = true)
 |-- net_amount_total: decimal(38,6) (nullable = true)
 |-- amount_difference: double (nullable = true)
 |-- liste_produits: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- liste_categories: array (nullable = true)
 |    |-- e

In [376]:
validation_results_parquet.printSchema()

root
 |-- controle: string (nullable = true)
 |-- nombre_anomalies: long (nullable = true)
 |-- statut: string (nullable = true)



In [377]:
table_rejets_parquet.printSchema()

root
 |-- source: string (nullable = true)
 |-- rejection_reason: string (nullable = true)
 |-- rejection_timestamp: timestamp (nullable = true)
 |-- original_data: string (nullable = true)



In [378]:
#Vérification des données

customer_order_360_parquet.show(5,truncate=False)


+--------------+-----------------+--------------+------------------+------------+----------+-------------+-------------------+------------+--------------+--------+----------------+------------------+--------------+------------------+--------------+----------------+-----------------+----------------------+----------------+-------------+-----------------------------+-----------+------------+--------+--------+-------------------------+---------------------+--------------------+------------------+-------------------+-------------------+-------------------+--------------------+------------------+-------------+----------+-----------+
|order_id_clean|customer_id_clean|full_name     |email_clean       |phone_clean |city_clean|country_clean|order_date_clean   |order_status|payment_method|currency|total_amount_eur|number_of_products|total_quantity|gross_amount_total|discount_total|net_amount_total|amount_difference|liste_produits        |liste_categories|liste_marques|nombre_categories_different

In [379]:
validation_results_parquet.show(truncate=False)

+------------------------------+----------------+------+
|controle                      |nombre_anomalies|statut|
+------------------------------+----------------+------+
|Commandes annulées livrées    |2               |ERREUR|
|Livrées sans date de livraison|0               |OK    |
|Notes hors intervalle [1,5]   |0               |OK    |
|Scores qualité invalides      |0               |OK    |
|Livraison avant commande      |0               |OK    |
|Statuts non normalisés        |0               |OK    |
|Unicité des commandes         |0               |OK    |
|Montants négatifs             |0               |OK    |
+------------------------------+----------------+------+



In [380]:
table_rejets_parquet.show(5,truncate=False)

+------+--------------------------+--------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|source|rejection_reason          |rejection_timestamp       |original_data                                                                                                                                                                                                                                                                                                                                                

In [381]:
#Contrôle de cohérence

print("Volume original :",customer_order_360.count())

print("Volume relu :",customer_order_360_parquet.count())

Volume original : 14
Volume relu : 14


In [382]:
#Unicité des commandes

customer_order_360_parquet.groupBy("order_id_clean").count().filter(col("count") > 1).show()

+--------------+-----+
|order_id_clean|count|
+--------------+-----+
+--------------+-----+



In [383]:
#Nombre de partitions

print("Nombre de partitions :",customer_order_360_parquet.rdd.getNumPartitions())

Nombre de partitions : 1


In [384]:
#Années disponibles

customer_order_360_parquet.select("order_year").distinct().show()

+----------+
|order_year|
+----------+
|      2026|
+----------+



In [385]:
#Mois disponibles

customer_order_360_parquet.select("order_month").distinct().show()

+-----------+
|order_month|
+-----------+
|          3|
+-----------+



In [386]:
#Présence des colonnes demandées

print(customer_order_360_parquet.columns)

['order_id_clean', 'customer_id_clean', 'full_name', 'email_clean', 'phone_clean', 'city_clean', 'country_clean', 'order_date_clean', 'order_status', 'payment_method', 'currency', 'total_amount_eur', 'number_of_products', 'total_quantity', 'gross_amount_total', 'discount_total', 'net_amount_total', 'amount_difference', 'liste_produits', 'liste_categories', 'liste_marques', 'nombre_categories_differentes', 'nombre_avis', 'note_moyenne', 'note_min', 'note_max', 'pourcentage_avis_positifs', 'customer_satisfaction', 'last_delivery_status', 'carrier_name_clean', 'first_event', 'last_event', 'delivery_delay_days', 'delivery_performance', 'data_quality_score', 'quality_level', 'order_year', 'order_month']


## Partie 15 — Analyses avec Spark SQL


#### Étape 39 : création d’une vue temporaire

In [387]:
customer_order_360_parquet.createOrReplaceTempView("customer_order_360_view")

In [388]:
spark.sql("""
SELECT *
FROM customer_order_360_view
LIMIT 5
""").show(truncate=False)

+--------------+-----------------+--------------+------------------+------------+----------+-------------+-------------------+------------+--------------+--------+----------------+------------------+--------------+------------------+--------------+----------------+-----------------+----------------------+----------------+-------------+-----------------------------+-----------+------------+--------+--------+-------------------------+---------------------+--------------------+------------------+-------------------+-------------------+-------------------+--------------------+------------------+-------------+----------+-----------+
|order_id_clean|customer_id_clean|full_name     |email_clean       |phone_clean |city_clean|country_clean|order_date_clean   |order_status|payment_method|currency|total_amount_eur|number_of_products|total_quantity|gross_amount_total|discount_total|net_amount_total|amount_difference|liste_produits        |liste_categories|liste_marques|nombre_categories_different

#### Étape 40 : requêtes analytiques

In [389]:
#1. Chiffre d'affaires mensuel par pays
spark.sql("""
SELECT
    country_clean,
    order_year,
    order_month,
    ROUND(SUM(total_amount_eur),2) AS chiffre_affaires
FROM customer_order_360_view
GROUP BY country_clean, order_year, order_month
ORDER BY country_clean, order_year, order_month
""").show(truncate=False)

+-------------+----------+-----------+----------------+
|country_clean|order_year|order_month|chiffre_affaires|
+-------------+----------+-----------+----------------+
|NULL         |2026      |3          |9.25            |
|France       |2026      |3          |795.01          |
+-------------+----------+-----------+----------------+



In [390]:
#2. Top 5 catégories générant le plus de CA
spark.sql("""
SELECT
    categorie,
    ROUND(SUM(total_amount_eur),2) AS chiffre_affaires
FROM (
    SELECT
        explode(liste_categories) AS categorie,
        total_amount_eur
    FROM customer_order_360_view
)
GROUP BY categorie
ORDER BY chiffre_affaires DESC
LIMIT 5
""").show(truncate=False)

+-----------+----------------+
|categorie  |chiffre_affaires|
+-----------+----------------+
|Technologie|270.93          |
+-----------+----------------+



In [391]:
#3. Top 10 clients ayant le plus dépensé
spark.sql("""
SELECT
    customer_id_clean,
    full_name,
    ROUND(SUM(total_amount_eur),2) AS montant_total
FROM customer_order_360_view
GROUP BY customer_id_clean, full_name
ORDER BY montant_total DESC
LIMIT 10
""").show(truncate=False)

+-----------------+--------------+-------------+
|customer_id_clean|full_name     |montant_total|
+-----------------+--------------+-------------+
|C000007          |Prenom7 Nom7  |201.0        |
|C000010          |Prenom10 Nom10|109.75       |
|C000009          |Prenom9 Nom9  |100.5        |
|C000016          |Prenom16 Nom16|100.5        |
|C000004          |Prenom4 Nom4  |100.5        |
|C000015          |Prenom15 Nom15|55.9         |
|C000027          |Prenom27 Nom27|55.9         |
|C000002          |Prenom2 Nom2  |51.43        |
|C000999          |NULL          |9.25         |
|C000022          |Prenom22 Nom22|9.25         |
+-----------------+--------------+-------------+



In [392]:
#4. Note moyenne par catégorie
spark.sql("""
SELECT
    categorie,
    ROUND(AVG(note_moyenne),2) AS note_moyenne_categorie
FROM (
    SELECT
        explode(liste_categories) AS categorie,
        note_moyenne
    FROM customer_order_360_view
    WHERE note_moyenne IS NOT NULL
)
GROUP BY categorie
ORDER BY note_moyenne_categorie DESC
""").show(truncate=False)

+-----------+----------------------+
|categorie  |note_moyenne_categorie|
+-----------+----------------------+
|Technologie|3.0                   |
+-----------+----------------------+



In [393]:
#5. Transporteur avec le délai moyen le plus faible
spark.sql("""
SELECT
    carrier_name_clean,
    ROUND(AVG(delivery_delay_days),2) AS delai_moyen
FROM customer_order_360_view
WHERE delivery_delay_days IS NOT NULL
GROUP BY carrier_name_clean
ORDER BY delai_moyen ASC
""").show(truncate=False)

+------------------+-----------+
|carrier_name_clean|delai_moyen|
+------------------+-----------+
|Dhl               |11.43      |
|Ups               |13.0       |
+------------------+-----------+



In [394]:
#6. Pourcentage de commandes livrées en moins de 3 jours
spark.sql("""
SELECT
    ROUND(
        100.0 *
        SUM(CASE
            WHEN delivery_delay_days < 3 THEN 1
            ELSE 0
            END)
        /
        COUNT(*)
    ,2) AS pourcentage_livraison_rapide
FROM customer_order_360_view
WHERE delivery_delay_days IS NOT NULL
""").show(truncate=False)

+----------------------------+
|pourcentage_livraison_rapide|
+----------------------------+
|18.18                       |
+----------------------------+



In [395]:
#7. Clients ayant au moins 3 commandes et une note moyenne < 3

spark.sql("""
SELECT
    customer_id_clean,
    full_name,
    COUNT(*) AS nb_commandes,
    ROUND(AVG(note_moyenne), 2) AS note_moyenne_client
FROM customer_order_360_view
WHERE note_moyenne IS NOT NULL
GROUP BY customer_id_clean, full_name
HAVING COUNT(*) >= 3
   AND AVG(note_moyenne) < 3
ORDER BY note_moyenne_client
""").show(truncate=False)


+-----------------+---------+------------+-------------------+
|customer_id_clean|full_name|nb_commandes|note_moyenne_client|
+-----------------+---------+------------+-------------------+
+-----------------+---------+------------+-------------------+



In [396]:
#8. Commandes dont le montant déclaré diffère du montant recalculé
spark.sql("""
SELECT
    order_id_clean,
    total_amount_eur,
    net_amount_total,
    amount_difference
FROM customer_order_360_view
WHERE amount_difference > 0.01
ORDER BY amount_difference DESC
""").show(truncate=False)

+--------------+----------------+----------------+-----------------+
|order_id_clean|total_amount_eur|net_amount_total|amount_difference|
+--------------+----------------+----------------+-----------------+
|ORD000007     |51.43           |285.000000      |233.57           |
|ORD000020     |100.5           |160.000000      |59.5             |
|ORD000004     |100.5           |45.000000       |55.5             |
|ORD000013     |9.25            |62.500000       |53.25            |
|ORD000036     |9.25            |25.000000       |15.75            |
+--------------+----------------+----------------+-----------------+



In [397]:
#9. Pourcentage de commandes orphelines
spark.sql("""
SELECT
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN full_name IS NULL THEN 1
                ELSE 0
            END
        ) / COUNT(*)
    ,2) AS pourcentage_commandes_orphelines
FROM customer_order_360_view
""").show(truncate=False)

+--------------------------------+
|pourcentage_commandes_orphelines|
+--------------------------------+
|7.14                            |
+--------------------------------+



In [398]:
#10. Répartition des commandes selon le niveau de qualité
spark.sql("""
SELECT
    quality_level,
    COUNT(*) AS nb_commandes,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM customer_order_360_view),
        2
    ) AS pourcentage
FROM customer_order_360_view
GROUP BY quality_level
ORDER BY nb_commandes DESC
""").show(truncate=False)


+-------------+------------+-----------+
|quality_level|nb_commandes|pourcentage|
+-------------+------------+-----------+
|Excellent    |9           |64.29      |
|Bon          |3           |21.43      |
|Moyen        |2           |14.29      |
+-------------+------------+-----------+



## Partie 16 — Optimisation Spark

#### Étape 41 : analyse du plan d’exécution

In [399]:
#Jointure 1 : Clients + Commandes

jointure_clients_commandes.explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (58)
+- Project (57)
   +- SortMergeJoin LeftOuter (56)
      :- Sort (17)
      :  +- Exchange (16)
      :     +- Project (15)
      :        +- BroadcastHashJoin LeftOuter BuildRight (14)
      :           :- Project (10)
      :           :  +- Project (9)
      :           :     +- Project (8)
      :           :        +- Project (7)
      :           :           +- Project (6)
      :           :              +- Project (5)
      :           :                 +- Project (4)
      :           :                    +- Project (3)
      :           :                       +- Filter (2)
      :           :                          +- Scan JDBCRelation(orders) [numPartitions=1]  (1)
      :           +- BroadcastExchange (13)
      :              +- Filter (12)
      :                 +- Scan ExistingRDD (11)
      +- Project (55)
         +- Filter (54)
            +- Window (53)
               +- WindowGroupLimit (52)
                  +- Sort (

In [400]:
#Jointure 2 : Commandes + Produits

jointure_commandes_produits.explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (104)
+- Project (103)
   +- SortMergeJoin LeftOuter (102)
      :- Project (74)
      :  +- SortMergeJoin LeftOuter (73)
      :     :- Sort (59)
      :     :  +- Exchange (58)
      :     :     +- Project (57)
      :     :        +- SortMergeJoin LeftOuter (56)
      :     :           :- Sort (17)
      :     :           :  +- Exchange (16)
      :     :           :     +- Project (15)
      :     :           :        +- BroadcastHashJoin LeftOuter BuildRight (14)
      :     :           :           :- Project (10)
      :     :           :           :  +- Project (9)
      :     :           :           :     +- Project (8)
      :     :           :           :        +- Project (7)
      :     :           :           :           +- Project (6)
      :     :           :           :              +- Project (5)
      :     :           :           :                 +- Project (4)
      :     :           :           :                    +- Project 

#### Étape 42 : utilisation du broadcast

In [401]:
from pyspark.sql.functions import broadcast

#Taux de change
orders_step15 = (
    orders_step14
    .join(
        broadcast(exchange_rates),
        on="currency",
        how="left"))

In [402]:
orders_step15.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (16)
+- Project (15)
   +- BroadcastHashJoin LeftOuter BuildRight (14)
      :- Project (10)
      :  +- Project (9)
      :     +- Project (8)
      :        +- Project (7)
      :           +- Project (6)
      :              +- Project (5)
      :                 +- Project (4)
      :                    +- Project (3)
      :                       +- Filter (2)
      :                          +- Scan JDBCRelation(orders) [numPartitions=1]  (1)
      +- BroadcastExchange (13)
         +- Filter (12)
            +- Scan ExistingRDD (11)


(1) Scan JDBCRelation(orders) [numPartitions=1] 
Output [7]: [order_id#7679, customer_id#7680, order_date#7681, status#7682, payment_method#7683, total_amount#7684, currency#7685]
ReadSchema: struct<order_id:string,customer_id:string,order_date:string,status:string,payment_method:string,total_amount:decimal(38,18),currency:string>

(2) Filter
Input [7]: [order_id#7679, customer_id#7680, order_date#7681, status#

In [403]:
#Dictionnaire des pays

dictionnaire_pays = spark.createDataFrame(
    [
        ("FR", "France"),
        ("ES", "Espagne"),
        ("MA", "Maroc"),
        ("UK", "Royaume-Uni")
    ],
    ["country_code", "country_name"])

In [404]:
jointure_pays_sans_broadcast = (
    customers_clean.join(
        dictionnaire_pays,
        customers_clean.country_clean ==
        dictionnaire_pays.country_name,
        "left"))

In [405]:
jointure_pays_sans_broadcast.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (46)
+- SortMergeJoin LeftOuter (45)
   :- Sort (40)
   :  +- Exchange (39)
   :     +- Project (38)
   :        +- Filter (37)
   :           +- Window (36)
   :              +- WindowGroupLimit (35)
   :                 +- Sort (34)
   :                    +- Exchange (33)
   :                       +- WindowGroupLimit (32)
   :                          +- Sort (31)
   :                             +- Project (30)
   :                                +- Project (29)
   :                                   +- Project (28)
   :                                      +- Project (27)
   :                                         +- Project (26)
   :                                            +- Project (25)
   :                                               +- SortMergeJoin LeftOuter (24)
   :                                                  :- Sort (12)
   :                                                  :  +- Exchange (11)
   :                        

In [406]:
jointure_pays_avec_broadcast = (
    customers_clean.join(
        broadcast(dictionnaire_pays),
        customers_clean.country_clean ==
        dictionnaire_pays.country_name,
        "left"))

In [407]:
jointure_pays_avec_broadcast.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (43)
+- BroadcastHashJoin LeftOuter BuildRight (42)
   :- Project (38)
   :  +- Filter (37)
   :     +- Window (36)
   :        +- WindowGroupLimit (35)
   :           +- Sort (34)
   :              +- Exchange (33)
   :                 +- WindowGroupLimit (32)
   :                    +- Sort (31)
   :                       +- Project (30)
   :                          +- Project (29)
   :                             +- Project (28)
   :                                +- Project (27)
   :                                   +- Project (26)
   :                                      +- Project (25)
   :                                         +- SortMergeJoin LeftOuter (24)
   :                                            :- Sort (12)
   :                                            :  +- Exchange (11)
   :                                            :     +- Project (10)
   :                                            :        +- Project (9)
   :        

In [408]:
#Dictionnaire des catégories

dictionnaire_categories = spark.createDataFrame(
    [
        ("Technologie", "TECH")
    ],
    ["category_name", "category_code"])


In [409]:
categories_sans_broadcast = (
    products_clean.join(
        dictionnaire_categories,
        products_clean.category_clean ==
        dictionnaire_categories.category_name,
        "left"))

In [410]:
categories_sans_broadcast.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (20)
+- SortMergeJoin LeftOuter (19)
   :- Sort (14)
   :  +- Exchange (13)
   :     +- Project (12)
   :        +- Filter (11)
   :           +- Window (10)
   :              +- WindowGroupLimit (9)
   :                 +- Sort (8)
   :                    +- Exchange (7)
   :                       +- WindowGroupLimit (6)
   :                          +- Sort (5)
   :                             +- Project (4)
   :                                +- Project (3)
   :                                   +- Project (2)
   :                                      +- Scan JDBCRelation(products) [numPartitions=1]  (1)
   +- Sort (18)
      +- Exchange (17)
         +- Filter (16)
            +- Scan ExistingRDD (15)


(1) Scan JDBCRelation(products) [numPartitions=1] 
Output [6]: [product_id#7775, product_name#7776, category#7777, brand#7778, current_price#7779, active#7780]
ReadSchema: struct<product_id:string,product_name:string,category:string,brand:string

In [411]:
categories_avec_broadcast = (
    products_clean.join(
        broadcast(dictionnaire_categories),
        products_clean.category_clean ==
        dictionnaire_categories.category_name,
        "left"))

In [412]:
categories_avec_broadcast.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (17)
+- BroadcastHashJoin LeftOuter BuildRight (16)
   :- Project (12)
   :  +- Filter (11)
   :     +- Window (10)
   :        +- WindowGroupLimit (9)
   :           +- Sort (8)
   :              +- Exchange (7)
   :                 +- WindowGroupLimit (6)
   :                    +- Sort (5)
   :                       +- Project (4)
   :                          +- Project (3)
   :                             +- Project (2)
   :                                +- Scan JDBCRelation(products) [numPartitions=1]  (1)
   +- BroadcastExchange (15)
      +- Filter (14)
         +- Scan ExistingRDD (13)


(1) Scan JDBCRelation(products) [numPartitions=1] 
Output [6]: [product_id#7775, product_name#7776, category#7777, brand#7778, current_price#7779, active#7780]
ReadSchema: struct<product_id:string,product_name:string,category:string,brand:string,current_price:decimal(38,18),active:boolean>

(2) Project
Output [7]: [product_id#7775, product_name#7776, cate

In [413]:
#Table Produits

print(products_clean.count())

15


In [414]:
jointure_produits_broadcast = (
    order_items_step16
    .join(
        broadcast(products_clean),
        on="product_id_clean",
        how="left"))

In [415]:
jointure_produits_broadcast.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (25)
+- Project (24)
   +- BroadcastHashJoin LeftOuter BuildRight (23)
      :- Project (8)
      :  +- Project (7)
      :     +- Project (6)
      :        +- Project (5)
      :           +- Project (4)
      :              +- Project (3)
      :                 +- Filter (2)
      :                    +- Scan JDBCRelation(order_items) [numPartitions=1]  (1)
      +- BroadcastExchange (22)
         +- Project (21)
            +- Filter (20)
               +- Window (19)
                  +- WindowGroupLimit (18)
                     +- Sort (17)
                        +- Exchange (16)
                           +- WindowGroupLimit (15)
                              +- Sort (14)
                                 +- Project (13)
                                    +- Project (12)
                                       +- Project (11)
                                          +- Filter (10)
                                             +- Scan JDBCR

La comparaison des plans d'exécution montre que les jointures classiques utilisent généralement des opérations SortMergeJoin, Sort et Exchange.

Après activation du broadcast, Spark utilise principalement des opérations BroadcastHashJoin, ce qui réduit les échanges réseau et améliore les performances d'exécution.

#### Étape 43 : gestion du cache

In [416]:
import time

debut = time.time()

customer_order_360.count()

temps_sans_cache = time.time() - debut

print("Temps sans cache :", temps_sans_cache)

Temps sans cache : 0.37899303436279297


In [417]:
customer_order_360.cache()

customer_order_360.count()

14

In [418]:
# Mesure

debut = time.time()

customer_order_360.count()

temps_avec_cache = time.time() - debut

print("Temps avec cache :", temps_avec_cache)

Temps avec cache : 0.24803519248962402


In [419]:
customer_order_360.unpersist()

DataFrame[order_id_clean: string, customer_id_clean: string, full_name: string, email_clean: string, phone_clean: string, city_clean: string, country_clean: string, order_date_clean: timestamp, order_status: string, payment_method: string, currency: string, total_amount_eur: double, number_of_products: bigint, total_quantity: bigint, gross_amount_total: decimal(38,7), discount_total: decimal(38,6), net_amount_total: decimal(38,6), amount_difference: double, liste_produits: array<string>, liste_categories: array<string>, liste_marques: array<string>, nombre_categories_differentes: bigint, nombre_avis: bigint, note_moyenne: double, note_min: int, note_max: int, pourcentage_avis_positifs: double, customer_satisfaction: string, last_delivery_status: string, carrier_name_clean: string, first_event: timestamp, last_event: timestamp, delivery_delay_days: int, delivery_performance: string, data_quality_score: int, quality_level: string]

In [420]:
debut = time.time()

customer_order_360.count()

temps_apres_unpersist = time.time() - debut

print("Temps après unpersist :", temps_apres_unpersist)

Temps après unpersist : 0.2594282627105713


In [421]:
print("Sans cache :", temps_sans_cache)
print("Avec cache :", temps_avec_cache)
print("Après unpersist :", temps_apres_unpersist)

Sans cache : 0.37899303436279297
Avec cache : 0.24803519248962402
Après unpersist : 0.2594282627105713


In [422]:
print(
    "Customer 360 en cache :",
    customer_order_360.is_cached)

Customer 360 en cache : False


Le DataFrame 
customer_order_360 a été mis en cache car il est réutilisé plusieurs fois dans le pipeline.

La comparaison des temps d'exécution montre qu'une action exécutée après la mise en cache est généralement plus rapide car Spark réutilise les données présentes en mémoire.

Après l'appel à unpersist(), Spark supprime le DataFrame du cache et doit recalculer les transformations lors des actions suivantes.

Tous les DataFrames ne doivent pas être mis en cache car :

- le cache consomme de la mémoire ;
- certains DataFrames ne sont utilisés qu'une seule fois ;
- un excès de cache peut dégrader les performances globales du cluster.

Le cache doit être réservé aux DataFrames coûteux à recalculer et réutilisés plusieurs fois.

#### Étape 44 : gestion des partitions

In [423]:
print(
    "Partitions actuelles :",
    customer_order_360.rdd.getNumPartitions())

Partitions actuelles : 1


In [424]:
customer_order_360_repartition = (
    customer_order_360
    .repartition(4))

In [425]:
print(
    "Partitions après repartition :",
    customer_order_360_repartition.rdd.getNumPartitions())

Partitions après repartition : 4


In [426]:
import time

debut = time.time()

customer_order_360_repartition.write \
    .mode("overwrite") \
    .parquet("output/test_repartition")

temps_repartition = time.time() - debut

print(
    "Temps repartition :",
    temps_repartition)

Temps repartition : 0.5330331325531006


In [427]:
customer_order_360_coalesce = (
    customer_order_360
    .coalesce(1))

In [428]:
print(
    "Partitions après coalesce :",
    customer_order_360_coalesce.rdd.getNumPartitions())

Partitions après coalesce : 1


In [429]:
debut = time.time()

customer_order_360_coalesce.write \
    .mode("overwrite") \
    .parquet("output/test_coalesce")

temps_coalesce = time.time() - debut

print(
    "Temps coalesce :",
    temps_coalesce)

Temps coalesce : 0.5387792587280273


In [431]:
import os

print(
    "Fichiers repartition :",
    len(os.listdir("output/test_repartition")))

print(
    "Fichiers coalesce :",
    len(os.listdir("output/test_coalesce")))

Fichiers repartition : 10
Fichiers coalesce : 4


In [432]:
def taille_dossier(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(
                os.path.join(root, f)
            )
    return total

print(
    "Taille repartition :",
    taille_dossier("output/test_repartition"),
    "octets"
)

print(
    "Taille coalesce :",
    taille_dossier("output/test_coalesce"),
    "octets")

Taille repartition : 46780 octets
Taille coalesce : 12811 octets


In [433]:
print("Temps repartition :", temps_repartition)
print("Temps coalesce :", temps_coalesce)

Temps repartition : 0.5330331325531006
Temps coalesce : 0.5387792587280273


# Partie 17 – Questions de réflexion

1. Pourquoi faut-il normaliser les identifiants avant les jointures ?

Les mêmes identifiants peuvent être représentés sous différents formats (par exemple : `ORD1`, `ORD001`, `ord-001`). La normalisation permet d'obtenir une représentation unique et garantit des jointures correctes entre les différentes sources.

2. Quelle différence existe-t-il entre dropDuplicates() et une déduplication avec Window ?

La fonction `dropDuplicates()` conserve une ligne de manière arbitraire parmi les doublons. Une déduplication basée sur une fonction de fenêtrage (`Window`) permet d'appliquer une règle métier précise, par exemple conserver l'enregistrement le plus récent ou celui disposant de la meilleure qualité de données.

3. Pourquoi utiliser une jointure gauche lors de la construction de la vue finale ?

Une jointure gauche permet de conserver toutes les commandes même lorsque certaines informations sont absentes dans d'autres sources. Cela évite de perdre des données importantes et facilite la détection des anomalies.

4. Pourquoi les montants financiers doivent-ils utiliser DecimalType ?

Les types flottants (`FloatType` ou `DoubleType`) peuvent introduire des erreurs d'arrondi. `DecimalType` offre une précision contrôlée indispensable pour les calculs financiers et comptables.

5. Quels sont les risques liés à l’inférence automatique du schéma JSON ?

L'inférence automatique peut attribuer des types incorrects, produire des schémas incohérents d'un fichier à l'autre, ralentir la lecture des données et générer des valeurs nulles inattendues.

6. Pourquoi faut-il agréger les lignes de commande avant certaines jointures ?

Une commande peut contenir plusieurs lignes de détail. L'agrégation permet de produire des indicateurs au niveau de la commande et d'éviter la duplication artificielle des données lors des jointures.

7. Comment éviter une multiplication artificielle des lignes ?

Il faut agréger les données avant les jointures, utiliser des clés de jointure adaptées et privilégier des fonctions comme `collect_set()` ou `countDistinct()` lorsque cela est nécessaire.

8. Dans quel cas une jointure broadcast devient-elle dangereuse ?

Une jointure broadcast devient problématique lorsque la table diffusée est trop volumineuse. Elle peut alors saturer la mémoire des exécuteurs et provoquer une dégradation importante des performances.

9. Quelle différence existe-t-il entre repartition() et coalesce() ?

`repartition()` redistribue complètement les données et provoque un shuffle. `coalesce()` réduit le nombre de partitions en limitant les mouvements de données. `coalesce()` est généralement plus efficace avant l'écriture des résultats.

10. Pourquoi conserver les données rejetées ?

Les données rejetées permettent d'assurer la traçabilité, de faciliter les analyses de qualité de données et de corriger ultérieurement les anomalies sans perdre l'information d'origine.

11. Comment rendre ce traitement incrémental ?

Un traitement incrémental peut être mis en œuvre en ne traitant que les nouvelles données depuis la dernière exécution, par exemple à l'aide d'une date de chargement ou d'un mécanisme de Change Data Capture (CDC).

12. Comment garantir l’idempotence du traitement ?

Un traitement est idempotent lorsqu'une exécution répétée produit toujours le même résultat. Cela peut être garanti grâce à des clés métier uniques, des transformations déterministes et des écritures contrôlées.

13. Comment traiter l’arrivée tardive d’un événement de livraison ?

Il est nécessaire de conserver l'historique des événements et de recalculer les indicateurs concernés lors de l'arrivée de nouvelles informations afin de maintenir la cohérence des données.

14. Quel format pourrait remplacer Parquet afin de gérer les mises à jour et les opérations MERGE ?

Des formats tels que Delta Lake, Apache Iceberg ou Apache Hudi permettent de gérer les opérations MERGE, les mises à jour, les suppressions et le versionnement des données.

15. Quels indicateurs faudrait-il superviser en production ?

Les principaux indicateurs à surveiller sont :
- le volume de données traité ;
- le taux de rejet ;
- le temps d'exécution des traitements ;
- le nombre d'erreurs ;
- le nombre de commandes orphelines ;
- les écarts financiers détectés ;
- la qualité des données ;
- le délai moyen de livraison ;
- la disponibilité des sources de données.